# 04 — Locked Benchmark, Final Refit, and Audited Submission

This notebook is the first stage permitted to inspect the locked 2022–2025 tournament outcomes and the 2026 Stage 2 matchup matrix. It begins only after `03_model_comparison_and_diagnostics.ipynb` has completed its self-contained integrity gate and frozen `configs/model_recipe.yaml`.

## Experimental boundary

This notebook does **not** reopen feature engineering, architecture selection, model-family selection, hyperparameter search, calibration-family selection, probability guarding, or ensemble-weight selection. Those decisions were made using development seasons ending in 2021.

It evaluates three predeclared prediction products for each tournament:

1. **Primary** — the one-standard-error recommendation from notebook 03.
2. **Performance challenger** — the lowest development Brier-score stream from notebook 03.
3. **Development blend** — a nonnegative primary/challenger blend learned exclusively from development out-of-fold predictions.

Two historical protocols are reported:

- **Prequential refit:** refit the frozen procedure using all legally available seasons before each locked year.
- **Static block:** fit through 2021 and hold the fitted model fixed across 2022–2025.

The primary submission remains tied to the predeclared primary streams regardless of the locked benchmark result. Challenger and blend submissions are preserved as transparent sensitivity products rather than post-benchmark replacements.

## Main outputs

- immutable locked-benchmark predictions and comprehensive metrics;
- season-, tournament-, stage-, seed-gap-, and confidence-level diagnostics;
- season-clustered bootstrap comparisons;
- interactive Plotly reports;
- final refits through 2025;
- seed-aware and seed-free scoring routes;
- three fully audited 2026 submission files;
- exact bracket advancement probabilities when official 2026 seeds and slots are available;
- a model card, artifact manifest, and benchmark-consumption lock.


## 1. Environment, contracts, and notebook 03 handoff

The first code cell verifies the complete upstream chain and refuses to continue unless:

- notebooks 01–03 report successful completion;
- all 180 model-comparison tasks completed without failure;
- logistic regression, XGBoost, LightGBM, PyTorch, and TensorFlow completed in the requested separate and pooled comparisons;
- all required metrics, ablations, explanations, and Plotly reports passed;
- `metric_integrity_issues.csv` is empty;
- the authoritative development recipe is `configs/model_recipe.yaml`;
- the feature-store and model-contract hashes still match;
- the locked seasons and Stage 2 were not used during model development.


In [ ]:
from __future__ import annotations

# Conservative CPU limits are set before importing compiled libraries.
import os
for _name in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ.setdefault(_name, "4")

import gc
import hashlib
import json
import logging
import math
import platform
import random
import sys
import time
import traceback
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import psutil
import pyarrow.parquet as pq
import scipy
import sklearn
import xgboost as xgb
import yaml
from plotly.subplots import make_subplots
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

try:
    from march_mania.paths import get_project_paths
except Exception as exc:
    raise ImportError(
        "The local march_mania package is unavailable. From the repository root run "
        "`python -m pip install -e . --no-deps`, restart the kernel, and rerun."
    ) from exc

warnings_to_filter = (
    r"(?s).*Found Intel OpenMP.*LLVM OpenMP.*",
)
import warnings
warnings.filterwarnings("once", category=RuntimeWarning)
for _pattern in warnings_to_filter:
    warnings.filterwarnings("ignore", message=_pattern, category=RuntimeWarning)

SEED = 2026
NOTEBOOK_IMPLEMENTATION_VERSION = 3
random.seed(SEED)
np.random.seed(SEED)

PATHS = get_project_paths()
ROOT = Path(PATHS.root)
INTERIM = Path(PATHS.interim)
PROCESSED = Path(PATHS.processed)
CONFIG_DIR = ROOT / "configs"
REPORT_DIR = ROOT / "reports" / "final_2026"
FIGURE_DIR = ROOT / "reports" / "figures" / "final_2026"
CACHE_DIR = ROOT / "data" / "model_cache" / "04_locked_benchmark_final"
MODEL_DIR = ROOT / "models" / "final_2026"
SUBMISSION_DIR = ROOT / "submissions"
LOG_DIR = REPORT_DIR / "logs"
for _directory in (
    CONFIG_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    CACHE_DIR,
    MODEL_DIR,
    SUBMISSION_DIR,
    LOG_DIR,
):
    _directory.mkdir(parents=True, exist_ok=True)

PROCESS = psutil.Process(os.getpid())
PHYSICAL_CORES = psutil.cpu_count(logical=False) or 2
MAX_THREADS = max(1, min(4, PHYSICAL_CORES))
RAM_GB = psutil.virtual_memory().total / 1024**3
RUN_LOG = LOG_DIR / "run.log"
EVENT_LOG = LOG_DIR / "events.jsonl"


def log_event(event: str, **details: Any) -> None:
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "event": event,
        **details,
    }
    with EVENT_LOG.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, default=str) + "\n")
    with RUN_LOG.open("a", encoding="utf-8") as handle:
        handle.write(
            f"{record['timestamp_utc']} | {event} | "
            f"{json.dumps(details, default=str)}\n"
        )


def sha256_file(path: Path, chunk_bytes: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


def object_sha256(value: Any) -> str:
    payload = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def atomic_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def atomic_json(path: Path, value: Any) -> None:
    atomic_text(path, json.dumps(value, indent=2, default=str))


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_parquet(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(temporary, index=False, compression="zstd")
    os.replace(temporary, path)


def atomic_joblib(path: Path, value: Any, compress: int = 3) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    joblib.dump(value, temporary, compress=compress)
    os.replace(temporary, path)


def rss_mb() -> float:
    return PROCESS.memory_info().rss / 1024**2


def clip_probability(values: Any, epsilon: float = 1e-6) -> np.ndarray:
    return np.clip(
        np.asarray(values, dtype=np.float64),
        epsilon,
        1.0 - epsilon,
    )


def downcast_numeric(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    for column in output.select_dtypes(include=["float64"]).columns:
        output[column] = output[column].astype("float32")
    for column in output.select_dtypes(include=["int64"]).columns:
        if column in {"Season", "Team1ID", "Team2ID"}:
            continue
        values = output[column]
        if not values.notna().all():
            continue
        minimum, maximum = values.min(), values.max()
        if np.iinfo(np.int16).min <= minimum <= maximum <= np.iinfo(np.int16).max:
            output[column] = values.astype("int16")
        elif np.iinfo(np.int32).min <= minimum <= maximum <= np.iinfo(np.int32).max:
            output[column] = values.astype("int32")
    return output


def ensure_memory(label: str, minimum_available_gb: float = 0.45) -> dict[str, float]:
    memory = psutil.virtual_memory()
    available = memory.available / 1024**3
    snapshot = {
        "available_gb": float(available),
        "available_percent": float(memory.available / memory.total * 100),
        "rss_mb": float(rss_mb()),
    }
    if available < minimum_available_gb:
        raise MemoryError(
            f"MEMORY PRESSURE PAUSE before {label}: {available:.2f} GB available; "
            f"{minimum_available_gb:.2f} GB required. Completed checkpoints are preserved."
        )
    if available < 1.0:
        print(
            f"WARNING | Low system RAM before {label}: {available:.2f} GB available. "
            "Execution remains sequential and checkpointed."
        )
    return snapshot


def save_plotly(figure: go.Figure, name: str) -> str:
    path = FIGURE_DIR / f"{name}.html"
    figure.write_html(
        path,
        include_plotlyjs="directory",
        full_html=True,
    )
    figure.show()
    return path.name


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def choose_existing(paths: Sequence[Path], label: str) -> Path:
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Missing {label}. Checked: {[str(path) for path in paths]}"
    )


def as_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(False)
    )


SPLIT_PATH = CONFIG_DIR / "splits.yaml"
FEATURE_PATH = CONFIG_DIR / "features.yaml"
MODEL_CONFIG_PATH = CONFIG_DIR / "modeling.yaml"
RECIPE_PATH = CONFIG_DIR / "model_recipe.yaml"
READINESS_01_PATH = ROOT / "reports" / "modeling" / "01_readiness_summary.json"
READINESS_02_PATH = ROOT / "reports" / "feature_engineering" / "02_readiness_summary.json"
REPORT_03 = ROOT / "reports" / "modeling" / "03_model_comparison"
READINESS_03_PATH = REPORT_03 / "03_readiness_summary.json"
FINALIZATION_03_PATH = REPORT_03 / "03_finalization_summary.json"
FINAL_CHECKS_03_PATH = REPORT_03 / "final_checks.csv"
WINNERS_03_PATH = REPORT_03 / "winner_summary.json"
ARTIFACTS_03_PATH = REPORT_03 / "artifact_manifest.csv"
METRIC_ISSUES_03_PATH = REPORT_03 / "metric_integrity_issues.csv"
FRAMEWORK_COVERAGE_03_PATH = REPORT_03 / "requested_framework_coverage.csv"
MATCHED_COMPARISON_03_PATH = REPORT_03 / "matched_season_model_comparison.csv"
CANDIDATE_PATH = ROOT / "reports" / "feature_engineering" / "candidate_feature_sets.json"
REGISTRY_PATH = choose_existing(
    [
        ROOT / "reports" / "feature_engineering" / "feature_registry_v1.csv",
        ROOT / "reports" / "feature_engineering" / "feature_registry.csv",
    ],
    "feature registry",
)
HISTORICAL_PATH = PROCESSED / "historical_matchup_feature_store_v1.parquet"
STAGE2_PATH = PROCESSED / "stage2_matchup_feature_store_v1.parquet"
LOCKED_MANIFEST_PATH = INTERIM / "fold_manifest_locked_benchmark.parquet"
SUBMISSION_ROUTING_PATH = INTERIM / "submission_routing.parquet"
DEVELOPMENT_OOF_PATH = PROCESSED / "development_oof_predictions_model_comparison.parquet"

required_paths = [
    SPLIT_PATH,
    FEATURE_PATH,
    MODEL_CONFIG_PATH,
    RECIPE_PATH,
    READINESS_01_PATH,
    READINESS_02_PATH,
    READINESS_03_PATH,
    FINALIZATION_03_PATH,
    FINAL_CHECKS_03_PATH,
    WINNERS_03_PATH,
    ARTIFACTS_03_PATH,
    METRIC_ISSUES_03_PATH,
    FRAMEWORK_COVERAGE_03_PATH,
    MATCHED_COMPARISON_03_PATH,
    CANDIDATE_PATH,
    REGISTRY_PATH,
    HISTORICAL_PATH,
    STAGE2_PATH,
    LOCKED_MANIFEST_PATH,
    DEVELOPMENT_OOF_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, (
    "Required upstream artifacts are missing. Complete notebook 03 finalization first: "
    f"{missing_paths}"
)

SPLITS = yaml.safe_load(SPLIT_PATH.read_text(encoding="utf-8"))
FEATURE_CONFIG = yaml.safe_load(FEATURE_PATH.read_text(encoding="utf-8"))
MODEL_CONFIG = yaml.safe_load(MODEL_CONFIG_PATH.read_text(encoding="utf-8"))
MODEL_RECIPE = yaml.safe_load(RECIPE_PATH.read_text(encoding="utf-8"))
READINESS_01 = read_json(READINESS_01_PATH)
READINESS_02 = read_json(READINESS_02_PATH)
READINESS_03 = read_json(READINESS_03_PATH)
FINALIZATION_03 = read_json(FINALIZATION_03_PATH)
FINAL_CHECKS_03 = pd.read_csv(FINAL_CHECKS_03_PATH)
WINNERS_03 = read_json(WINNERS_03_PATH)
ARTIFACTS_03 = pd.read_csv(ARTIFACTS_03_PATH)
try:
    METRIC_ISSUES_03 = pd.read_csv(METRIC_ISSUES_03_PATH)
except pd.errors.EmptyDataError:
    METRIC_ISSUES_03 = pd.DataFrame()
FRAMEWORK_COVERAGE_03 = pd.read_csv(FRAMEWORK_COVERAGE_03_PATH)
MATCHED_COMPARISON_03 = pd.read_csv(MATCHED_COMPARISON_03_PATH)
CANDIDATE_SETS: dict[str, list[str]] = read_json(CANDIDATE_PATH)
FEATURE_REGISTRY = pd.read_csv(REGISTRY_PATH).drop_duplicates("Feature")
LOCKED_MANIFEST = pd.read_parquet(LOCKED_MANIFEST_PATH)

assert READINESS_01["status"] == "complete"
assert READINESS_02["status"] == "complete"
assert READINESS_02["blocking_feature_check_failures"] == 0
assert READINESS_02["symmetry_failures"] == 0
assert READINESS_03["status"] == "complete"
assert int(READINESS_03.get("notebook_implementation_version", 0)) >= 5
assert READINESS_03["planned_core_tasks"] == READINESS_03["completed_core_tasks"] == 180
assert READINESS_03["failed_core_tasks"] == 0
assert READINESS_03["registered_failures"] == 0
assert READINESS_03["requested_framework_coverage_complete"] is True
assert READINESS_03["completed_explanation_pairs"] == READINESS_03["expected_explanation_pairs"] == 10
assert READINESS_03["blocking_final_check_failures"] == 0
assert READINESS_03["recipe_status"] == "frozen_development_recipe"
assert READINESS_03["stage2_loaded"] is False
assert READINESS_03["locked_benchmark_evaluated"] is False
assert FINALIZATION_03["status"] == "complete"
assert int(FINALIZATION_03.get("metric_integrity_issues", 0)) == 0
assert METRIC_ISSUES_03.empty
assert as_bool_series(FRAMEWORK_COVERAGE_03["Complete"]).all()
assert as_bool_series(
    FINAL_CHECKS_03.loc[
        as_bool_series(FINAL_CHECKS_03["Blocking"]),
        "Passed",
    ]
).all()
assert MODEL_RECIPE["status"] == "frozen_development_recipe"
assert MODEL_RECIPE["locked_benchmark_evaluated"] is False
assert MODEL_RECIPE["recipe_sha256"] == READINESS_03["recipe_sha256"]
assert MODEL_RECIPE["source_contracts"]["split"] == SPLITS["contract_sha256"]
assert MODEL_RECIPE["source_contracts"]["feature"] == FEATURE_CONFIG["feature_contract_sha256"]
assert MODEL_RECIPE["source_contracts"]["model"] == MODEL_CONFIG["model_contract_sha256"]
assert sha256_file(HISTORICAL_PATH) == READINESS_02["artifact_sha256"][
    "historical_matchup_feature_store_v1"
]

oof_manifest_row = ARTIFACTS_03.loc[ARTIFACTS_03["Artifact"].eq("development_oof")]
if not oof_manifest_row.empty:
    assert sha256_file(DEVELOPMENT_OOF_PATH) == str(oof_manifest_row.iloc[0]["SHA256"])

LOCKED_SEASONS = tuple(sorted(map(int, SPLITS["locked_benchmark_seasons"])))
TARGET_SEASON = int(SPLITS["target_season"])
DEVELOPMENT_LAST_SEASON = int(
    SPLITS.get("development_last_season", MODEL_CONFIG["development_last_season"])
)
EXPECTED_CONTRACTS = {
    "split": SPLITS["contract_sha256"],
    "feature": FEATURE_CONFIG["feature_contract_sha256"],
    "model": MODEL_CONFIG["model_contract_sha256"],
    "recipe": MODEL_RECIPE["recipe_sha256"],
    "historical_store": READINESS_02["artifact_sha256"][
        "historical_matchup_feature_store_v1"
    ],
    "stage2_store": READINESS_02["artifact_sha256"][
        "stage2_matchup_feature_store_v1"
    ],
    "development_oof": sha256_file(DEVELOPMENT_OOF_PATH),
}

if os.environ.get("MM_ALLOW_NON_ML_KERNEL", "0") != "1":
    assert "ml-modeling" in str(sys.executable).lower(), (
        "Select the Python (ml-modeling) kernel before continuing."
    )

print("Project root:", ROOT)
print("Notebook implementation version:", NOTEBOOK_IMPLEMENTATION_VERSION)
print("Python:", sys.executable)
print("Python version:", platform.python_version())
print("NumPy / pandas / scikit-learn:", np.__version__, pd.__version__, sklearn.__version__)
print("XGBoost / LightGBM:", xgb.__version__, lgb.__version__)
print("Physical cores / model threads:", PHYSICAL_CORES, "/", MAX_THREADS)
print("System RAM GB / process RSS MB:", round(RAM_GB, 2), "/", round(rss_mb(), 2))
print("Finalized notebook 03 recipe:", RECIPE_PATH)
print("Locked seasons:", LOCKED_SEASONS)
print("Target season:", TARGET_SEASON)

log_event(
    "notebook_started",
    contracts=EXPECTED_CONTRACTS,
    locked_seasons=LOCKED_SEASONS,
    target_season=TARGET_SEASON,
    notebook_implementation_version=NOTEBOOK_IMPLEMENTATION_VERSION,
    notebook03_implementation_version=READINESS_03.get("notebook_implementation_version"),
    package_versions={
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "lightgbm": lgb.__version__,
    },
)


## 2. Development handoff and neural-model audit

Notebook 03 has already completed model selection. This section summarizes the frozen development evidence before any locked outcome is read.

The table distinguishes:

- the one-standard-error **primary** stream;
- the lowest-Brier **performance challenger**;
- PyTorch and TensorFlow tabular MLP results in separate and pooled architectures.

The neural rows are diagnostic evidence, not candidates for promotion after the benchmark is opened. Notebook 04 preserves the recipe frozen in notebook 03.


In [ ]:
PLOT_ARTIFACTS: list[str] = []

development_handoff = MATCHED_COMPARISON_03.copy()
required_handoff_columns = {
    "Gender",
    "Architecture",
    "Model",
    "ModelDisplay",
    "MacroSeasonBrier",
    "GameWeightedBrier",
    "ROCAUC",
    "AveragePrecision",
    "AUCPR",
    "Precision",
    "Recall",
    "F1",
    "CalibrationSlope",
    "ECE",
}
missing_handoff_columns = sorted(
    required_handoff_columns.difference(development_handoff.columns)
)
assert not missing_handoff_columns, (
    "Notebook 03 comparison report is missing columns: "
    f"{missing_handoff_columns}"
)

decision_rows = []
for gender_key, gender_label in (("men", "M"), ("women", "W")):
    for decision_key, decision_label in (
        ("recommended", "Primary — one-standard-error"),
        ("raw_best", "Performance challenger — lowest Brier"),
    ):
        selection = WINNERS_03[gender_key][decision_key]
        decision_rows.append(
            {
                "Gender": gender_label,
                "Decision": decision_label,
                "Architecture": selection["architecture"],
                "Model": selection["model"],
                "ModelDisplay": selection.get(
                    "model_display",
                    selection["model"],
                ),
                "MacroSeasonBrier": float(
                    selection["macro_season_brier"]
                ),
                "ROCAUC": float(selection["roc_auc"]),
                "AveragePrecision": float(
                    selection["average_precision"]
                ),
                "Precision": float(selection["precision"]),
                "Recall": float(selection["recall"]),
                "F1": float(selection["f1"]),
                "CalibrationSlope": float(
                    selection["calibration_slope"]
                ),
                "ECE": float(selection["ece"]),
            }
        )
development_decisions = pd.DataFrame(decision_rows)

neural_development = (
    development_handoff.loc[
        development_handoff["Model"].isin(
            ["torch_mlp", "tensorflow_mlp"]
        ),
        [
            "Gender",
            "Architecture",
            "Model",
            "ModelDisplay",
            "MacroSeasonBrier",
            "GameWeightedBrier",
            "ROCAUC",
            "AveragePrecision",
            "AUCPR",
            "Precision",
            "Recall",
            "F1",
            "CalibrationSlope",
            "ECE",
        ],
    ]
    .sort_values(
        ["Gender", "MacroSeasonBrier", "Architecture"]
    )
    .reset_index(drop=True)
)

atomic_csv(
    REPORT_DIR / "development_handoff_decisions.csv",
    development_decisions,
)
atomic_csv(
    REPORT_DIR / "development_neural_model_audit.csv",
    neural_development,
)

display(development_decisions)
display(neural_development)

figure = px.bar(
    development_handoff.loc[
        development_handoff["Gender"].isin(["M", "W"])
    ].sort_values(
        ["Gender", "MacroSeasonBrier"]
    ),
    x="MacroSeasonBrier",
    y="ModelDisplay",
    color="Architecture",
    facet_col="Gender",
    facet_col_spacing=0.08,
    orientation="h",
    hover_data=[
        "ROCAUC",
        "AveragePrecision",
        "AUCPR",
        "Precision",
        "Recall",
        "F1",
        "CalibrationSlope",
        "ECE",
    ],
    title=(
        "Development handoff: matched-season Brier scores "
        "before opening the locked benchmark"
    ),
    labels={
        "MacroSeasonBrier": "Mean held-out season Brier",
        "ModelDisplay": "Model",
    },
    height=900,
)
figure.update_yaxes(matches=None, showticklabels=True)
PLOT_ARTIFACTS.append(
    save_plotly(figure, "development_handoff_scorecard")
)

print(
    "Notebook 03 handoff verified. Locked outcomes and Stage 2 "
    "remain unopened."
)


## 3. Production modeling and evaluation engine

The implementation below reproduces the fold-local feature governance used in notebook 03, enforces algebraic matchup symmetry, supports the selected linear and tree model families, and computes both probability and threshold-based diagnostics.

Feature selection is fitted only on the available training seasons. Imputation, scaling, feature ranking, correlation pruning, and model fitting never inspect the season being predicted.


In [ ]:
# Feature governance, symmetric prediction, model fitting, calibration, and metrics.
REGISTRY_BY_FEATURE = FEATURE_REGISTRY.set_index("Feature").to_dict(orient="index")
CANDIDATE_UNION = sorted(
    set().union(*(set(values) for values in CANDIDATE_SETS.values()))
)


def underlying_feature_name(column: str) -> str:
    for prefix in ("diff__", "absdiff__", "mean__", "max__", "min__"):
        if column.startswith(prefix):
            return column[len(prefix):]
    return column


def infer_feature_block(column: str) -> str:
    if column in REGISTRY_BY_FEATURE:
        return str(REGISTRY_BY_FEATURE[column].get("Block", "unregistered"))
    base = underlying_feature_name(column)
    if base in REGISTRY_BY_FEATURE:
        return str(REGISTRY_BY_FEATURE[base].get("Block", "unregistered"))
    if column.startswith("seedprior__"):
        return "prequential_seed_priors"
    if column.startswith("matchup__seed") or "team1_seed" in column or "team2_seed" in column:
        return "selection_committee_prior"
    if column.startswith("interaction__"):
        return "matchup_interactions"
    if column.startswith(("context__", "availability__")):
        return "availability_and_context"
    token_map = {
        "massey__": "massey_consensus",
        "coach__": "prior_coach_history",
        "prior__": "prior_program_history",
        "schedule__": "schedule_and_accomplishment",
        "conference__": "conference_context",
        "adj__": "opponent_adjusted_efficiency",
        "rating__": "dynamic_ratings",
        "detailed__": "detailed_efficiency",
        "compact__": "compact_performance",
        "norm__": "within_season_normalization",
        "geo__": "geography",
    }
    for token, block in token_map.items():
        if token in column:
            return block
    return "other"


FEATURE_TO_BLOCK = {
    feature: infer_feature_block(feature)
    for feature in CANDIDATE_UNION
}
BLOCK_BASE_QUOTAS = {
    "selection_committee_prior": 14,
    "prequential_seed_priors": 10,
    "compact_performance": 30,
    "dynamic_ratings": 24,
    "global_strength_ratings": 20,
    "schedule_and_accomplishment": 22,
    "conference_context": 10,
    "detailed_efficiency": 34,
    "opponent_adjusted_efficiency": 18,
    "prior_program_history": 10,
    "prior_coach_history": 7,
    "massey_consensus": 24,
    "matchup_interactions": 12,
    "within_season_normalization": 18,
    "availability_and_context": 10,
    "geography": 8,
    "other": 8,
}
MANDATORY_TOKEN_GROUPS = [
    ("matchup__seed_diff",),
    ("rating__elo_mov_538", "rating__elo_mov_log", "rating__elo_standard"),
    ("compact__margin_trim02", "compact__margin_mean"),
    ("schedule__opponent_strength_mean", "schedule__rpi"),
    ("adj__net_rtg_a50p0", "adj__net_rtg_a10p0"),
    ("massey__strength_mean", "massey__rank_mean"),
]
DIRECTIONAL_INTERACTION_TOKENS = ("_edge", "squared_signed")


def first_matching_feature(
    candidates: Sequence[str],
    tokens: Sequence[str],
) -> str | None:
    for token in tokens:
        matches = [
            feature
            for feature in candidates
            if feature == token or feature.endswith(token) or token in feature
        ]
        directional = [
            feature
            for feature in matches
            if feature.startswith(("diff__", "matchup__", "interaction__"))
        ]
        if directional:
            return sorted(directional)[0]
        if matches:
            return sorted(matches)[0]
    return None


def mandatory_features(
    candidates: Sequence[str],
    *,
    pooled: bool,
    seed_aware: bool,
) -> list[str]:
    selected: list[str] = []
    for tokens in MANDATORY_TOKEN_GROUPS:
        if not seed_aware and any("seed" in token for token in tokens):
            continue
        match = first_matching_feature(candidates, tokens)
        if match is not None:
            selected.append(match)
    if pooled and "context__women" in candidates:
        selected.append("context__women")
    return list(dict.fromkeys(selected))


def baseline_features(
    model_name: str,
    candidates: Sequence[str],
    *,
    pooled: bool,
) -> list[str]:
    if model_name == "seed_logistic":
        ordered = [
            feature
            for feature in candidates
            if feature
            in {
                "matchup__seed_diff",
                "matchup__seed_gap_abs",
                "matchup__team1_is_seed_favorite",
                "matchup__equal_seed",
            }
            or feature.startswith("seedprior__")
        ]
        preferred = [
            feature
            for feature in ordered
            if any(
                token in feature
                for token in (
                    "matchup__seed_diff",
                    "matchup__seed_gap_abs",
                    "team1_is_seed_favorite",
                    "pc8",
                    "overall",
                )
            )
        ]
        selected = preferred[:12] if preferred else ordered[:12]
    elif model_name == "elo_seed_logistic":
        selected = baseline_features(
            "seed_logistic",
            candidates,
            pooled=pooled,
        )
        selected += sorted(
            [
                feature
                for feature in candidates
                if any(
                    token in feature
                    for token in (
                        "rating__elo_mov_538",
                        "rating__elo_mov_log",
                        "rating__elo_standard",
                    )
                )
                and feature.startswith(("diff__", "absdiff__"))
            ]
        )[:12]
    else:
        selected = []
    if pooled and "context__women" in candidates:
        selected.append("context__women")
    return list(dict.fromkeys(selected))


def safe_column_medians(matrix: np.ndarray) -> np.ndarray:
    output = np.zeros(matrix.shape[1], dtype=np.float64)
    for index in range(matrix.shape[1]):
        values = matrix[:, index]
        finite = values[np.isfinite(values)]
        output[index] = float(np.median(finite)) if finite.size else 0.0
    return output


def vectorized_correlations(
    frame: pd.DataFrame,
    features: Sequence[str],
    target: np.ndarray,
) -> np.ndarray:
    if not features:
        return np.array([], dtype=np.float64)
    matrix = np.array(
        frame[list(features)].to_numpy(
            dtype=np.float64,
            na_value=np.nan,
        ),
        dtype=np.float64,
        copy=True,
        order="C",
    )
    medians = safe_column_medians(matrix)
    missing = ~np.isfinite(matrix)
    if missing.any():
        matrix[missing] = np.take(medians, np.where(missing)[1])
    matrix -= matrix.mean(axis=0, keepdims=True)
    centered_target = target.astype(np.float64) - np.mean(target)
    numerator = matrix.T @ centered_target
    denominator = np.sqrt(
        np.sum(matrix * matrix, axis=0)
        * np.sum(centered_target * centered_target)
    )
    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0,
    )


@dataclass
class SelectionResult:
    selected_features: list[str]
    audit: pd.DataFrame
    effective_cap: int
    selector_hash: str


def fit_block_aware_selector(
    training: pd.DataFrame,
    candidates: Sequence[str],
    *,
    pooled: bool,
    seed_aware: bool,
    model_name: str,
) -> SelectionResult:
    candidates = [
        feature
        for feature in candidates
        if feature in training.columns
        and pd.api.types.is_numeric_dtype(training[feature])
    ]
    if not seed_aware:
        candidates = [
            feature
            for feature in candidates
            if "seed" not in feature.lower()
        ]
    assert candidates, "No numeric candidates are available."

    target = training["Team1Win"].to_numpy(dtype=np.float64)
    missingness = training[candidates].isna().mean()
    nonmissing = training[candidates].notna().sum()
    unique_counts = training[candidates].nunique(dropna=True)
    fs_config = MODEL_CONFIG["feature_selection"]
    eligible = [
        feature
        for feature in candidates
        if missingness[feature] <= float(fs_config["missingness_max"])
        and nonmissing[feature]
        >= min(int(fs_config["minimum_nonmissing_rows"]), len(training))
        and unique_counts[feature] > 1
    ]
    mandatory = [
        feature
        for feature in mandatory_features(
            candidates,
            pooled=pooled,
            seed_aware=seed_aware,
        )
        if unique_counts.get(feature, 0) > 1
    ]
    eligible = list(dict.fromkeys(mandatory + eligible))
    assert eligible, "Every candidate was removed by training-only filters."

    global_corr = vectorized_correlations(training, eligible, target)
    per_group: dict[str, list[float]] = {
        feature: [] for feature in eligible
    }
    group_columns = ["Season", "Gender"] if pooled else ["Season"]
    grouper: str | list[str] = group_columns if pooled else "Season"
    for _, group in training.groupby(grouper, observed=True, sort=True):
        if len(group) < 20 or group["Team1Win"].nunique() < 2:
            continue
        correlations = vectorized_correlations(
            group,
            eligible,
            group["Team1Win"].to_numpy(dtype=np.float64),
        )
        for feature, value in zip(eligible, correlations, strict=True):
            if np.isfinite(value):
                per_group[feature].append(float(value))

    group_count = max(
        1,
        training[group_columns].drop_duplicates().shape[0],
    )
    records: list[dict[str, Any]] = []
    for index, feature in enumerate(eligible):
        correlations = np.asarray(
            per_group[feature],
            dtype=np.float64,
        )
        nonzero = correlations[np.abs(correlations) > 1e-12]
        sign_consistency = (
            abs(np.mean(np.sign(nonzero))) if nonzero.size else 0.0
        )
        coverage = len(correlations) / group_count
        median_abs = (
            float(np.median(np.abs(correlations)))
            if correlations.size
            else 0.0
        )
        iqr = (
            float(
                np.percentile(correlations, 75)
                - np.percentile(correlations, 25)
            )
            if correlations.size >= 2
            else 0.0
        )
        score = (
            0.48 * median_abs
            + 0.22 * abs(float(global_corr[index]))
            + 0.16 * sign_consistency
            + 0.10 * math.sqrt(max(coverage, 0.0))
            - 0.08 * float(missingness[feature])
            - 0.04 * min(iqr, 1.0)
        )
        records.append(
            {
                "Feature": feature,
                "Block": FEATURE_TO_BLOCK.get(feature, "other"),
                "MissingRate": float(missingness[feature]),
                "NonMissingRows": int(nonmissing[feature]),
                "UniqueValues": int(unique_counts[feature]),
                "GlobalCorrelation": float(global_corr[index]),
                "MedianAbsoluteSeasonCorrelation": median_abs,
                "SeasonSignConsistency": float(sign_consistency),
                "SeasonCoverage": float(coverage),
                "SeasonCorrelationIQR": iqr,
                "StabilityScore": float(score),
                "Mandatory": feature in mandatory,
            }
        )
    audit = pd.DataFrame(records).sort_values(
        ["Mandatory", "StabilityScore", "Feature"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    if model_name in {"elastic_logistic", "seed_logistic", "elo_seed_logistic"}:
        fixed_cap = int(fs_config["linear_feature_cap"])
        divisor = int(fs_config.get("row_adaptive_linear_divisor", 14))
    elif model_name in {"hist_classifier", "xgb_classifier", "lgb_classifier"}:
        fixed_cap = int(fs_config["tree_feature_cap"])
        divisor = int(fs_config.get("row_adaptive_tree_divisor", 8))
    else:
        fixed_cap = int(fs_config["linear_feature_cap"])
        divisor = 14
    cap = max(8, min(fixed_cap, max(12, len(training) // divisor)))
    pool_cap = min(300, max(2 * cap, cap + 20))
    quota_scale = cap / max(1, sum(BLOCK_BASE_QUOTAS.values()))
    block_choices: list[str] = []
    for block, group in audit.groupby("Block", observed=True, sort=False):
        base_quota = BLOCK_BASE_QUOTAS.get(block, 8)
        quota = max(
            2,
            min(
                base_quota,
                int(
                    math.ceil(
                        base_quota * max(0.65, quota_scale * 5)
                    )
                ),
            ),
        )
        block_choices.extend(group.head(quota)["Feature"].tolist())
    pool = list(
        dict.fromkeys(mandatory + block_choices + audit["Feature"].tolist())
    )[:pool_cap]

    matrix = np.array(
        training[pool].to_numpy(dtype=np.float32, na_value=np.nan),
        dtype=np.float32,
        copy=True,
        order="C",
    )
    medians = safe_column_medians(matrix).astype(np.float32)
    missing = ~np.isfinite(matrix)
    if missing.any():
        matrix[missing] = np.take(medians, np.where(missing)[1])
    standard_deviation = matrix.std(axis=0)
    normalized = (
        matrix - matrix.mean(axis=0, keepdims=True)
    ) / np.where(standard_deviation > 1e-12, standard_deviation, 1.0)
    correlation = np.abs(
        np.clip(
            (normalized.T @ normalized) / max(1, normalized.shape[0] - 1),
            -1,
            1,
        )
    )
    threshold = float(fs_config["correlation_prune_threshold"])
    selected: list[str] = []
    selected_indices: list[int] = []
    for index, feature in enumerate(pool):
        if feature in mandatory:
            selected.append(feature)
            selected_indices.append(index)
            continue
        if selected_indices and np.any(
            correlation[index, selected_indices] >= threshold
        ):
            continue
        selected.append(feature)
        selected_indices.append(index)
        if len(selected) >= cap:
            break
    if len(selected) > cap:
        mandatory_set = set(mandatory)
        selected = [
            feature for feature in selected if feature in mandatory_set
        ] + [
            feature for feature in selected if feature not in mandatory_set
        ][: max(0, cap - len(mandatory_set))]

    selected_set = set(selected)
    pool_set = set(pool)
    audit["InPrecorrelationPool"] = audit["Feature"].isin(pool_set)
    audit["Selected"] = audit["Feature"].isin(selected_set)
    audit["SelectionRank"] = audit["Feature"].map(
        {feature: rank + 1 for rank, feature in enumerate(selected)}
    )
    audit["ExclusionReason"] = np.select(
        [audit["Selected"], ~audit["InPrecorrelationPool"]],
        ["selected", "below_bounded_prescreen"],
        default="correlation_pruned_or_cap",
    )
    selector_hash = object_sha256(
        {
            "training_seasons": sorted(
                map(int, training["Season"].unique())
            ),
            "training_rows": len(training),
            "pooled": pooled,
            "seed_aware": seed_aware,
            "model": model_name,
            "candidate_hash": object_sha256(sorted(candidates)),
            "selected": selected,
            "contracts": EXPECTED_CONTRACTS,
        }
    )
    return SelectionResult(selected, audit, cap, selector_hash)


def writable_float32(
    frame: pd.DataFrame,
    features: Sequence[str],
) -> np.ndarray:
    return np.array(
        frame[list(features)].to_numpy(
            dtype=np.float32,
            na_value=np.nan,
        ),
        dtype=np.float32,
        copy=True,
        order="C",
    )


def mirror_features(
    frame: pd.DataFrame,
    features: Sequence[str],
    *,
    flip_targets: bool,
) -> pd.DataFrame:
    mirrored = frame.copy()
    feature_set = set(features)
    if {"matchup__team1_seed", "matchup__team2_seed"}.issubset(feature_set):
        mirrored[["matchup__team1_seed", "matchup__team2_seed"]] = frame[
            ["matchup__team2_seed", "matchup__team1_seed"]
        ].to_numpy(copy=True)
    equal_seed = (
        frame["matchup__equal_seed"].fillna(0).astype(bool)
        if "matchup__equal_seed" in frame.columns
        else pd.Series(False, index=frame.index)
    )
    for feature in features:
        if feature not in frame.columns or feature in {
            "matchup__team1_seed",
            "matchup__team2_seed",
        }:
            continue
        if feature.startswith("diff__") or feature == "matchup__seed_diff":
            mirrored[feature] = -pd.to_numeric(frame[feature], errors="coerce")
        elif feature.startswith("seedprior__") and "p_team1" in feature:
            mirrored[feature] = 1.0 - pd.to_numeric(
                frame[feature], errors="coerce"
            )
        elif feature == "matchup__team1_is_seed_favorite":
            original = pd.to_numeric(frame[feature], errors="coerce")
            mirrored[feature] = np.where(
                equal_seed,
                original,
                1.0 - original,
            )
        elif feature.startswith("interaction__") and any(
            token in feature for token in DIRECTIONAL_INTERACTION_TOKENS
        ):
            mirrored[feature] = -pd.to_numeric(frame[feature], errors="coerce")
    if flip_targets:
        if "Team1Win" in mirrored.columns:
            mirrored["Team1Win"] = 1 - mirrored["Team1Win"].astype(int)
        if "Team1Margin" in mirrored.columns:
            mirrored["Team1Margin"] = -pd.to_numeric(
                mirrored["Team1Margin"],
                errors="coerce",
            )
        if {"Team1ID", "Team2ID"}.issubset(mirrored.columns):
            mirrored[["Team1ID", "Team2ID"]] = frame[
                ["Team2ID", "Team1ID"]
            ].to_numpy(copy=True)
        if "TargetKey" in mirrored.columns:
            mirrored["TargetKey"] = (
                mirrored["TargetKey"].astype(str) + "__mirror"
            )
    return mirrored


@dataclass(frozen=True)
class ProductionModelSpec:
    name: str
    task: str
    raw_kind: str
    selector_family: str


PRODUCTION_MODEL_SPECS = {
    "seed_logistic": ProductionModelSpec(
        "seed_logistic", "classification", "probability", "baseline"
    ),
    "elo_seed_logistic": ProductionModelSpec(
        "elo_seed_logistic", "classification", "probability", "baseline"
    ),
    "elastic_logistic": ProductionModelSpec(
        "elastic_logistic", "classification", "probability", "linear"
    ),
    "hist_classifier": ProductionModelSpec(
        "hist_classifier", "classification", "probability", "tree"
    ),
    "xgb_classifier": ProductionModelSpec(
        "xgb_classifier", "classification", "probability", "tree"
    ),
    "lgb_classifier": ProductionModelSpec(
        "lgb_classifier", "classification", "probability", "tree"
    ),
}

DEFAULT_PARAMETERS: dict[str, dict[str, Any]] = {
    "seed_logistic": {"C": 0.50},
    "elo_seed_logistic": {"C": 0.50},
    "elastic_logistic": {"C": 0.10, "l1_ratio": 0.25},
    "hist_classifier": {
        "learning_rate": 0.04,
        "max_leaf_nodes": 15,
        "max_depth": 4,
        "min_samples_leaf": 25,
        "l2_regularization": 5.0,
        "max_iter": 300,
    },
    "xgb_classifier": {
        "eta": 0.04,
        "max_depth": 3,
        "min_child_weight": 8.0,
        "subsample": 0.85,
        "colsample_bytree": 0.70,
        "reg_alpha": 0.10,
        "reg_lambda": 12.0,
        "gamma": 0.05,
        "max_bin": 128,
    },
    "lgb_classifier": {
        "learning_rate": 0.03,
        "num_leaves": 15,
        "max_depth": 4,
        "min_data_in_leaf": 35,
        "feature_fraction": 0.70,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "lambda_l1": 0.10,
        "lambda_l2": 12.0,
        "min_gain_to_split": 0.02,
        "max_bin": 127,
    },
}


@dataclass
class FittedProductionModel:
    model_name: str
    model: Any
    preprocessor: Any
    features: list[str]
    best_iteration: int | None
    enforce_symmetry: bool = True

    def _predict_once(self, frame: pd.DataFrame) -> np.ndarray:
        matrix = writable_float32(frame, self.features)
        if self.model_name in {
            "seed_logistic",
            "elo_seed_logistic",
            "elastic_logistic",
        }:
            transformed = self.preprocessor.transform(matrix)
            return clip_probability(
                self.model.predict_proba(transformed)[:, 1]
            )
        if self.model_name == "hist_classifier":
            return clip_probability(self.model.predict_proba(matrix)[:, 1])
        if self.model_name == "xgb_classifier":
            data = xgb.DMatrix(matrix, feature_names=self.features)
            iteration = self.best_iteration or int(
                self.model.num_boosted_rounds()
            )
            return clip_probability(
                self.model.predict(
                    data,
                    iteration_range=(0, max(1, int(iteration))),
                )
            )
        if self.model_name == "lgb_classifier":
            return clip_probability(
                self.model.predict(
                    matrix,
                    num_iteration=self.best_iteration,
                )
            )
        raise KeyError(self.model_name)

    def predict_raw(self, frame: pd.DataFrame) -> np.ndarray:
        forward = self._predict_once(frame)
        if not self.enforce_symmetry:
            return forward
        reversed_frame = mirror_features(
            frame,
            self.features,
            flip_targets=False,
        )
        reverse = self._predict_once(reversed_frame)
        return clip_probability(0.5 * (forward + 1.0 - reverse))


def fit_production_model(
    training: pd.DataFrame,
    model_name: str,
    features: Sequence[str],
    params: dict[str, Any],
    random_seed: int,
    *,
    fixed_iterations: int | None,
) -> FittedProductionModel:
    features = list(features)
    assert features
    augmented = pd.concat(
        [
            training,
            mirror_features(training, features, flip_targets=True),
        ],
        ignore_index=True,
    )
    x_train = writable_float32(augmented, features)
    target = augmented["Team1Win"].to_numpy(dtype=int)

    if model_name in {
        "seed_logistic",
        "elo_seed_logistic",
        "elastic_logistic",
    }:
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        transformed = scaler.fit_transform(imputer.fit_transform(x_train))
        if model_name == "elastic_logistic":
            estimator = LogisticRegression(
                C=float(params["C"]),
                l1_ratio=float(params["l1_ratio"]),
                solver="saga",
                max_iter=12000,
                tol=1e-4,
                random_state=random_seed,
            )
        else:
            estimator = LogisticRegression(
                C=float(params["C"]),
                solver="lbfgs",
                max_iter=6000,
                random_state=random_seed,
            )
        estimator.fit(transformed, target)
        return FittedProductionModel(
            model_name,
            estimator,
            Pipeline([("imputer", imputer), ("scaler", scaler)]),
            features,
            None,
        )

    if model_name == "hist_classifier":
        estimator = HistGradientBoostingClassifier(
            **params,
            early_stopping=False,
            random_state=random_seed,
        )
        estimator.fit(x_train, target)
        return FittedProductionModel(
            model_name,
            estimator,
            None,
            features,
            int(params["max_iter"]),
        )

    if model_name == "xgb_classifier":
        num_rounds = int(fixed_iterations or 400)
        train_matrix = xgb.QuantileDMatrix(
            x_train,
            label=target,
            feature_names=features,
            max_bin=int(params.get("max_bin", 128)),
        )
        booster = xgb.train(
            {
                "objective": "binary:logistic",
                "tree_method": "hist",
                "device": "cpu",
                "nthread": MAX_THREADS,
                "seed": random_seed,
                "verbosity": 0,
                "disable_default_eval_metric": 1,
                **params,
            },
            train_matrix,
            num_boost_round=num_rounds,
            verbose_eval=False,
        )
        return FittedProductionModel(
            model_name,
            booster,
            None,
            features,
            num_rounds,
        )

    if model_name == "lgb_classifier":
        num_rounds = int(fixed_iterations or 500)
        train_set = lgb.Dataset(
            x_train,
            label=target,
            feature_name=features,
            free_raw_data=False,
        )
        booster = lgb.train(
            {
                "objective": "binary",
                "metric": "None",
                "verbosity": -1,
                "num_threads": MAX_THREADS,
                "seed": random_seed,
                "feature_fraction_seed": random_seed,
                "bagging_seed": random_seed,
                "data_random_seed": random_seed,
                "deterministic": True,
                "force_col_wise": True,
                **params,
            },
            train_set,
            num_boost_round=num_rounds,
            callbacks=[lgb.log_evaluation(period=0)],
        )
        return FittedProductionModel(
            model_name,
            booster,
            None,
            features,
            num_rounds,
        )
    raise KeyError(model_name)


class IdentityCalibrator:
    def predict(self, raw: np.ndarray) -> np.ndarray:
        return clip_probability(raw)


class ConstantCalibrator:
    def __init__(self, value: float):
        self.value = float(value)

    def predict(self, raw: np.ndarray) -> np.ndarray:
        return np.full(len(np.asarray(raw)), self.value)


class TemperatureCalibrator:
    def __init__(self, temperature: float):
        self.temperature = float(temperature)

    def predict(self, raw: np.ndarray) -> np.ndarray:
        return clip_probability(
            expit(logit(clip_probability(raw)) / self.temperature)
        )


class SklearnCalibrator:
    def __init__(
        self,
        estimator: Any,
        transform: Callable[[np.ndarray], np.ndarray],
    ):
        self.estimator = estimator
        self.transform = transform

    def predict(self, raw: np.ndarray) -> np.ndarray:
        return clip_probability(
            self.estimator.predict_proba(
                self.transform(np.asarray(raw))
            )[:, 1]
        )


class IsotonicCalibrator:
    def __init__(self, estimator: IsotonicRegression):
        self.estimator = estimator

    def predict(self, raw: np.ndarray) -> np.ndarray:
        return clip_probability(
            self.estimator.predict(np.asarray(raw, dtype=float))
        )


def probability_logit(raw: np.ndarray) -> np.ndarray:
    return logit(clip_probability(raw)).reshape(-1, 1)


def beta_transform(raw: np.ndarray) -> np.ndarray:
    probability = clip_probability(raw)
    return np.column_stack(
        [np.log(probability), -np.log1p(-probability)]
    )


def fit_calibrator(
    method: str,
    raw: np.ndarray,
    target: np.ndarray,
) -> Any:
    raw = np.asarray(raw, dtype=float)
    target = np.asarray(target, dtype=int)
    if np.unique(target).size < 2:
        return ConstantCalibrator(float(target.mean()))
    if method == "identity":
        return IdentityCalibrator()
    if method == "temperature":
        logits = logit(clip_probability(raw))
        result = minimize_scalar(
            lambda log_temperature: float(
                np.mean(
                    (
                        expit(logits / math.exp(log_temperature))
                        - target
                    )
                    ** 2
                )
            ),
            bounds=(math.log(0.20), math.log(5.0)),
            method="bounded",
        )
        return TemperatureCalibrator(math.exp(float(result.x)))
    if method == "platt":
        estimator = LogisticRegression(
            C=1e3,
            solver="lbfgs",
            max_iter=5000,
        ).fit(probability_logit(raw), target)
        return SklearnCalibrator(estimator, probability_logit)
    if method == "beta":
        estimator = LogisticRegression(
            C=1e3,
            solver="lbfgs",
            max_iter=5000,
        ).fit(beta_transform(raw), target)
        return SklearnCalibrator(estimator, beta_transform)
    if method == "spline":
        estimator = Pipeline(
            [
                (
                    "spline",
                    SplineTransformer(
                        n_knots=4,
                        degree=2,
                        include_bias=False,
                        extrapolation="linear",
                    ),
                ),
                (
                    "logistic",
                    LogisticRegression(
                        C=10.0,
                        solver="lbfgs",
                        max_iter=5000,
                    ),
                ),
            ]
        ).fit(probability_logit(raw), target)
        return SklearnCalibrator(estimator, probability_logit)
    if method == "isotonic":
        if len(raw) < 100 or np.unique(raw).size < 10:
            raise ValueError("Insufficient support for isotonic calibration.")
        estimator = IsotonicRegression(
            y_min=0,
            y_max=1,
            increasing=True,
            out_of_bounds="clip",
        ).fit(raw, target)
        return IsotonicCalibrator(estimator)
    raise KeyError(method)


def symmetric_calibration(calibrator: Any, raw: np.ndarray) -> np.ndarray:
    raw = clip_probability(raw)
    return clip_probability(
        0.5
        * (
            calibrator.predict(raw)
            + 1.0
            - calibrator.predict(1.0 - raw)
        )
    )


def apply_guard(
    probability: np.ndarray,
    shrinkage: float,
    floor: float,
) -> np.ndarray:
    return np.clip(
        0.5 + shrinkage * (np.asarray(probability) - 0.5),
        floor,
        1.0 - floor,
    )


def season_brier(
    frame: pd.DataFrame,
    column: str = "Prediction",
) -> pd.DataFrame:
    records = []
    for season, group in frame.groupby("Season", observed=True):
        records.append(
            {
                "Season": int(season),
                "Rows": int(len(group)),
                "Brier": float(
                    np.mean(
                        (
                            group[column].to_numpy(dtype=float)
                            - group["Team1Win"].to_numpy(dtype=float)
                        )
                        ** 2
                    )
                ),
            }
        )
    return pd.DataFrame(records)


def calibration_intercept_slope(
    target: np.ndarray,
    probability: np.ndarray,
) -> tuple[float, float]:
    target = np.asarray(target, dtype=int)
    probability = clip_probability(probability)
    if np.unique(target).size < 2 or np.unique(probability).size < 2:
        return np.nan, np.nan
    fitted = LogisticRegression(
        C=1e3,
        solver="lbfgs",
        max_iter=5000,
    ).fit(logit(probability).reshape(-1, 1), target)
    return float(fitted.intercept_[0]), float(fitted.coef_[0, 0])


def calibration_error(
    target: np.ndarray,
    probability: np.ndarray,
    bins: int = 10,
) -> tuple[float, float]:
    target = np.asarray(target, dtype=float)
    probability = clip_probability(probability)
    if len(probability) == 0:
        return np.nan, np.nan
    if np.unique(probability).size == 1:
        gap = abs(float(target.mean()) - float(probability[0]))
        return gap, gap
    identifiers = np.clip(
        np.digitize(probability, np.linspace(0, 1, bins + 1), right=True) - 1,
        0,
        bins - 1,
    )
    expected, maximum = 0.0, 0.0
    for index in range(bins):
        mask = identifiers == index
        if mask.any():
            gap = abs(
                float(probability[mask].mean())
                - float(target[mask].mean())
            )
            expected += float(mask.mean()) * gap
            maximum = max(maximum, gap)
    return float(expected), float(maximum)


def probability_metrics(
    frame: pd.DataFrame,
    column: str = "Prediction",
) -> dict[str, float]:
    """Probability and threshold diagnostics without single-class warnings.

    Discrimination and calibration-slope measures require both outcome
    classes. They are reported as NaN, rather than forcing a misleading
    value, for small diagnostic slices that contain only one class.
    """
    target = frame["Team1Win"].to_numpy(dtype=int)
    probability = clip_probability(frame[column])
    predicted = (probability >= 0.5).astype(int)
    by_season = season_brier(frame, column)
    has_both_classes = np.unique(target).size == 2

    if has_both_classes:
        precision_curve, recall_curve, _ = precision_recall_curve(
            target,
            probability,
        )
        pr_order = np.argsort(recall_curve)
        auc_pr = float(
            np.trapezoid(
                precision_curve[pr_order],
                recall_curve[pr_order],
            )
        )
        average_precision = float(
            average_precision_score(target, probability)
        )
        roc_auc = float(roc_auc_score(target, probability))
        balanced_accuracy = float(
            balanced_accuracy_score(target, predicted)
        )
    else:
        auc_pr = np.nan
        average_precision = np.nan
        roc_auc = np.nan
        balanced_accuracy = np.nan

    matrix = confusion_matrix(target, predicted, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    intercept, slope = calibration_intercept_slope(target, probability)
    ece, mce = calibration_error(target, probability)
    return {
        "Rows": int(len(frame)),
        "Seasons": int(frame["Season"].nunique()),
        "OutcomePrevalence": float(np.mean(target)),
        "ContainsBothOutcomeClasses": bool(has_both_classes),
        "MacroSeasonBrier": float(by_season["Brier"].mean()),
        "GameWeightedBrier": float(brier_score_loss(target, probability)),
        "SeasonBrierStd": (
            float(by_season["Brier"].std(ddof=1))
            if len(by_season) > 1
            else 0.0
        ),
        "WorstSeasonBrier": float(by_season["Brier"].max()),
        "LogLoss": float(log_loss(target, probability, labels=[0, 1])),
        "ROCAUC": roc_auc,
        "AveragePrecision": average_precision,
        "AUCPR": auc_pr,
        "Accuracy": float(accuracy_score(target, predicted)),
        "BalancedAccuracy": balanced_accuracy,
        "Precision": float(
            precision_score(target, predicted, zero_division=0)
        ),
        "Recall": float(recall_score(target, predicted, zero_division=0)),
        "F1": float(f1_score(target, predicted, zero_division=0)),
        "MCC": float(matthews_corrcoef(target, predicted)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "CalibrationIntercept": intercept,
        "CalibrationSlope": slope,
        "ECE": ece,
        "MCE": mce,
    }


CALIBRATOR_RANK = {
    "identity": 0,
    "temperature": 1,
    "platt": 2,
    "beta": 3,
    "spline": 4,
    "isotonic": 5,
}


@dataclass
class CalibrationSelection:
    method: str
    calibrator: Any
    cross_fitted: np.ndarray
    audit: pd.DataFrame


def select_calibration(oof: pd.DataFrame) -> CalibrationSelection:
    methods = list(MODEL_CONFIG["calibration"]["probability_methods"])
    seasons = sorted(map(int, oof["Season"].unique()))
    raw = oof["RawPrediction"].to_numpy(dtype=float)
    target = oof["Team1Win"].to_numpy(dtype=int)
    prediction_map: dict[str, np.ndarray] = {}
    records: list[dict[str, Any]] = []
    for method in methods:
        cross_fitted = np.full(len(oof), np.nan)
        fallback = False
        for season in seasons:
            validation_mask = oof["Season"].to_numpy(dtype=int) == season
            training_mask = ~validation_mask
            try:
                if training_mask.sum() < 40 or np.unique(target[training_mask]).size < 2:
                    raise ValueError("Insufficient calibration fold support.")
                fitted = fit_calibrator(
                    method,
                    raw[training_mask],
                    target[training_mask],
                )
                cross_fitted[validation_mask] = symmetric_calibration(
                    fitted,
                    raw[validation_mask],
                )
            except Exception:
                fallback = True
                cross_fitted[validation_mask] = clip_probability(
                    raw[validation_mask]
                )
        scored = oof[["Season", "Team1Win"]].copy()
        scored["Prediction"] = cross_fitted
        result = probability_metrics(scored)
        records.append(
            {
                "Method": method,
                "MacroSeasonBrier": result["MacroSeasonBrier"],
                "WorstSeasonBrier": result["WorstSeasonBrier"],
                "SeasonBrierStd": result["SeasonBrierStd"],
                "ComplexityRank": CALIBRATOR_RANK[method],
                "HadFallback": fallback,
            }
        )
        prediction_map[method] = cross_fitted
    audit = pd.DataFrame(records).sort_values(
        ["MacroSeasonBrier", "ComplexityRank"]
    ).reset_index(drop=True)
    best_method = str(audit.iloc[0]["Method"])
    best_scored = oof[["Season", "Team1Win"]].copy()
    best_scored["Prediction"] = prediction_map[best_method]
    best_by_season = season_brier(best_scored)
    standard_error = (
        float(
            best_by_season["Brier"].std(ddof=1)
            / math.sqrt(len(best_by_season))
        )
        if len(best_by_season) > 1
        else 0.0
    )
    threshold = float(audit.iloc[0]["MacroSeasonBrier"]) + standard_error
    chosen = audit.loc[
        audit["MacroSeasonBrier"] <= threshold + 1e-12
    ].sort_values(["ComplexityRank", "MacroSeasonBrier"]).iloc[0]
    method = str(chosen["Method"])
    audit["OneSEThreshold"] = threshold
    audit["WithinOneSE"] = (
        audit["MacroSeasonBrier"] <= threshold + 1e-12
    )
    audit["Selected"] = audit["Method"].eq(method)
    return CalibrationSelection(
        method,
        fit_calibrator(method, raw, target),
        prediction_map[method],
        audit,
    )


def select_probability_guard(
    oof: pd.DataFrame,
    calibrated: np.ndarray,
) -> tuple[dict[str, float], pd.DataFrame]:
    records = []
    for shrinkage in (0.80, 0.85, 0.90, 0.95, 1.00):
        for floor in (1e-6, 0.0025, 0.005, 0.01, 0.02, 0.025, 0.05):
            scored = oof[["Season", "Team1Win"]].copy()
            scored["Prediction"] = apply_guard(
                calibrated,
                shrinkage,
                floor,
            )
            result = probability_metrics(scored)
            records.append(
                {
                    "Shrinkage": shrinkage,
                    "ClipFloor": floor,
                    "MacroSeasonBrier": result["MacroSeasonBrier"],
                    "WorstSeasonBrier": result["WorstSeasonBrier"],
                    "IdentityDistance": abs(1.0 - shrinkage) + 2.0 * floor,
                }
            )
    audit = pd.DataFrame(records).sort_values(
        ["MacroSeasonBrier", "IdentityDistance"]
    ).reset_index(drop=True)
    best = audit.iloc[0]
    best_scored = oof[["Season", "Team1Win"]].copy()
    best_scored["Prediction"] = apply_guard(
        calibrated,
        float(best["Shrinkage"]),
        float(best["ClipFloor"]),
    )
    by_season = season_brier(best_scored)
    standard_error = (
        float(by_season["Brier"].std(ddof=1) / math.sqrt(len(by_season)))
        if len(by_season) > 1
        else 0.0
    )
    threshold = float(best["MacroSeasonBrier"]) + standard_error
    chosen = audit.loc[
        audit["MacroSeasonBrier"] <= threshold + 1e-12
    ].sort_values(["IdentityDistance", "MacroSeasonBrier"]).iloc[0]
    audit["OneSEThreshold"] = threshold
    audit["WithinOneSE"] = (
        audit["MacroSeasonBrier"] <= threshold + 1e-12
    )
    audit["Selected"] = (
        audit["Shrinkage"].eq(chosen["Shrinkage"])
        & audit["ClipFloor"].eq(chosen["ClipFloor"])
    )
    return {
        "shrinkage": float(chosen["Shrinkage"]),
        "clip_floor": float(chosen["ClipFloor"]),
    }, audit


print(
    "Production engine ready:",
    sorted(PRODUCTION_MODEL_SPECS),
)


## 4. Freeze the benchmark and production blueprint

Before loading a locked label, the notebook freezes:

- primary and challenger stream definitions;
- development-derived consensus hyperparameters and boosting lengths;
- calibration methods and probability guards;
- development-only blend weights;
- seed-free fallback procedures;
- batch size and memory policy;
- all upstream file hashes.

A pre-existing blueprint must match byte-for-byte. A changed blueprint cannot reuse an earlier benchmark lock.


In [ ]:
# Freeze stream definitions, development-derived parameters, calibration, and fallback routes.
SUPPORTED_STREAM_MODELS = set(PRODUCTION_MODEL_SPECS)


def normalized_selection(selection: dict[str, Any]) -> dict[str, Any]:
    return {
        "architecture": str(selection["architecture"]),
        "model": str(selection["model"]),
        "model_display": str(selection.get("model_display", selection["model"])),
        "macro_season_brier": float(selection["macro_season_brier"]),
    }


WINNER_SELECTIONS = {
    "M": {
        "primary": normalized_selection(WINNERS_03["men"]["recommended"]),
        "challenger": normalized_selection(WINNERS_03["men"]["raw_best"]),
    },
    "W": {
        "primary": normalized_selection(WINNERS_03["women"]["recommended"]),
        "challenger": normalized_selection(WINNERS_03["women"]["raw_best"]),
    },
}
for gender, roles in WINNER_SELECTIONS.items():
    for role, selection in roles.items():
        assert selection["model"] in SUPPORTED_STREAM_MODELS, (
            f"Notebook 04 does not support {gender}/{role} model "
            f"{selection['model']}."
        )


def training_gender(architecture: str, output_gender: str) -> str:
    return "Pooled" if architecture == "pooled_common" else output_gender


def candidate_set_for(
    architecture: str,
    output_gender: str,
    *,
    seed_aware: bool,
) -> str:
    prefix = (
        "pooled"
        if architecture == "pooled_common"
        else "men"
        if output_gender == "M"
        else "women"
    )
    suffix = "seed_aware" if seed_aware else "seed_free"
    key = f"{prefix}_rich_{suffix}"
    assert key in CANDIDATE_SETS, f"Missing candidate set: {key}"
    return key


STREAM_SPECS: dict[str, dict[str, Any]] = {}
for gender in ("M", "W"):
    for role in ("primary", "challenger"):
        selection = WINNER_SELECTIONS[gender][role]
        stream_id = f"{gender}_{role}"
        STREAM_SPECS[stream_id] = {
            "stream_id": stream_id,
            "output_gender": gender,
            "role": role,
            "architecture": selection["architecture"],
            "training_gender": training_gender(
                selection["architecture"],
                gender,
            ),
            "model": selection["model"],
            "model_display": selection["model_display"],
            "candidate_set": candidate_set_for(
                selection["architecture"],
                gender,
                seed_aware=True,
            ),
            "seed_aware": True,
            "development_macro_season_brier": selection[
                "macro_season_brier"
            ],
        }

# Seed-free fallbacks are operational completeness routes. They are not allowed
# to replace the seed-aware primary model for actual tournament games.
for gender in ("M", "W"):
    stream_id = f"{gender}_fallback"
    STREAM_SPECS[stream_id] = {
        "stream_id": stream_id,
        "output_gender": gender,
        "role": "fallback",
        "architecture": "separate_gender",
        "training_gender": gender,
        "model": "elastic_logistic",
        "model_display": "Seed-free elastic-net logistic",
        "candidate_set": candidate_set_for(
            "separate_gender",
            gender,
            seed_aware=False,
        ),
        "seed_aware": False,
        "development_macro_season_brier": None,
    }


METADATA_ROOT = (
    ROOT
    / "data"
    / "model_cache"
    / "03_model_comparison"
    / "full"
    / "outer_models"
)


def load_model_metadata() -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    if not METADATA_ROOT.exists():
        return records
    for path in METADATA_ROOT.glob("*/*/metadata.json"):
        try:
            record = read_json(path)
            if record.get("status", "complete") == "complete":
                record["metadata_path"] = str(path)
                records.append(record)
        except Exception:
            continue
    return records


MODEL_METADATA = load_model_metadata()


def consensus_value(values: Sequence[Any]) -> Any:
    usable = [value for value in values if value is not None]
    if not usable:
        return None
    if all(isinstance(value, (int, float, np.number)) for value in usable):
        median = float(np.median(np.asarray(usable, dtype=float)))
        if all(isinstance(value, (int, np.integer)) for value in usable):
            return int(round(median))
        return median
    counts = pd.Series(usable, dtype="object").value_counts(dropna=False)
    return counts.index[0]


def consensus_training_recipe(
    specification: dict[str, Any],
) -> tuple[dict[str, Any], int | None, dict[str, Any]]:
    model_name = specification["model"]
    default = dict(DEFAULT_PARAMETERS[model_name])
    matches = [
        record
        for record in MODEL_METADATA
        if record.get("model") == model_name
        and record.get("architecture") == specification["architecture"]
        and record.get("universe") == "rich"
        and record.get("gender") == specification["training_gender"]
        and int(record.get("validation_season", 9999))
        <= DEVELOPMENT_LAST_SEASON
        and isinstance(record.get("parameters"), dict)
    ]
    if matches:
        parameter_names = sorted(
            set().union(*(set(record["parameters"]) for record in matches))
        )
        parameters = dict(default)
        for name in parameter_names:
            value = consensus_value(
                [record["parameters"].get(name) for record in matches]
            )
            if value is not None:
                parameters[name] = value
        iteration_values = [
            record.get("fixed_iterations_from_inner")
            for record in matches
            if record.get("fixed_iterations_from_inner") is not None
        ]
        fixed_iterations = (
            int(round(float(np.median(iteration_values))))
            if iteration_values
            else None
        )
        provenance = {
            "source": "median_or_mode_of_nested_development_outer_folds",
            "records": len(matches),
            "metadata_paths": [record["metadata_path"] for record in matches],
        }
        return parameters, fixed_iterations, provenance
    return default, None, {
        "source": "pre_registered_default_missing_metadata",
        "records": 0,
    }


PARAMETERS_BY_STREAM: dict[str, dict[str, Any]] = {}
ITERATIONS_BY_STREAM: dict[str, int | None] = {}
PARAMETER_PROVENANCE: dict[str, dict[str, Any]] = {}
for stream_id, specification in STREAM_SPECS.items():
    parameters, iterations, provenance = consensus_training_recipe(
        specification
    )
    PARAMETERS_BY_STREAM[stream_id] = parameters
    ITERATIONS_BY_STREAM[stream_id] = iterations
    PARAMETER_PROVENANCE[stream_id] = provenance

route_candidate_union = sorted(
    set().union(
        *(
            set(CANDIDATE_SETS[specification["candidate_set"]])
            for specification in STREAM_SPECS.values()
        )
    )
)
historical_schema = set(pq.ParquetFile(HISTORICAL_PATH).schema.names)
HISTORY_METADATA = [
    "TargetKey",
    "GameKey",
    "Gender",
    "Season",
    "DayNum",
    "Team1ID",
    "Team2ID",
    "Team1Win",
    "Team1Margin",
    "DatasetRole",
    "RichUniverseEligible",
    "FeatureRoute",
]
history_columns = [
    column
    for column in HISTORY_METADATA + route_candidate_union
    if column in historical_schema
]
development = pd.read_parquet(
    HISTORICAL_PATH,
    columns=history_columns,
    filters=[("DatasetRole", "==", "development")],
)
development = downcast_numeric(development)
assert development["DatasetRole"].eq("development").all()
assert development["Season"].max() <= DEVELOPMENT_LAST_SEASON
assert not development["Season"].isin(LOCKED_SEASONS).any()

development_oof = pd.read_parquet(DEVELOPMENT_OOF_PATH)
assert development_oof["Season"].max() <= DEVELOPMENT_LAST_SEASON
assert not development_oof["Season"].isin(LOCKED_SEASONS).any()


def rows_for_stream(
    frame: pd.DataFrame,
    stream_id: str,
    seasons: Sequence[int] | None = None,
    *,
    output_only: bool = False,
) -> pd.DataFrame:
    specification = STREAM_SPECS[stream_id]
    mask = frame["RichUniverseEligible"].astype(bool)
    if seasons is not None:
        mask &= frame["Season"].isin(list(map(int, seasons)))
    if output_only or specification["training_gender"] != "Pooled":
        mask &= frame["Gender"].eq(specification["output_gender"])
    return frame.loc[mask].copy()


def choose_stream_features(
    training: pd.DataFrame,
    stream_id: str,
) -> tuple[list[str], SelectionResult | None]:
    specification = STREAM_SPECS[stream_id]
    candidates = [
        feature
        for feature in CANDIDATE_SETS[specification["candidate_set"]]
        if feature in training.columns
    ]
    if specification["model"] in {"seed_logistic", "elo_seed_logistic"}:
        features = baseline_features(
            specification["model"],
            candidates,
            pooled=specification["training_gender"] == "Pooled",
        )
        assert features
        return features, None
    selection = fit_block_aware_selector(
        training,
        candidates,
        pooled=specification["training_gender"] == "Pooled",
        seed_aware=bool(specification["seed_aware"]),
        model_name=specification["model"],
    )
    return selection.selected_features, selection


def fit_stream_core(
    history: pd.DataFrame,
    stream_id: str,
    training_seasons: Sequence[int],
    random_seed: int,
) -> tuple[FittedProductionModel, SelectionResult | None]:
    specification = STREAM_SPECS[stream_id]
    training = rows_for_stream(history, stream_id, training_seasons)
    assert not training.empty
    features, selection = choose_stream_features(training, stream_id)
    ensure_memory(
        f"fit {stream_id}",
        minimum_available_gb=(
            0.30
            if specification["model"] in {
                "seed_logistic",
                "elo_seed_logistic",
                "elastic_logistic",
            }
            else 0.45
        ),
    )
    fitted = fit_production_model(
        training,
        specification["model"],
        features,
        PARAMETERS_BY_STREAM[stream_id],
        random_seed,
        fixed_iterations=ITERATIONS_BY_STREAM[stream_id],
    )
    return fitted, selection


def selected_development_oof(stream_id: str) -> pd.DataFrame:
    specification = STREAM_SPECS[stream_id]
    output = development_oof.loc[
        development_oof["Universe"].eq("rich")
        & development_oof["Architecture"].eq(specification["architecture"])
        & development_oof["Model"].eq(specification["model"])
        & development_oof["Gender"].eq(specification["output_gender"])
    ].copy()
    assert not output.empty, f"No development OOF rows for {stream_id}."
    assert output[["OuterFoldID", "TargetKey"]].duplicated().sum() == 0
    return output


# Build development-only OOF predictions for the seed-free fallback routes.
FALLBACK_OOF_PATH = CACHE_DIR / "development_seed_free_fallback_oof.parquet"
FALLBACK_META_PATH = CACHE_DIR / "development_seed_free_fallback_oof.json"
fallback_validation_seasons = list(
    map(int, MODEL_CONFIG["matched_outer_validation_seasons"])
)
fallback_signature = object_sha256(
    {
        "contracts": EXPECTED_CONTRACTS,
        "streams": {
            stream_id: STREAM_SPECS[stream_id]
            for stream_id in ("M_fallback", "W_fallback")
        },
        "parameters": {
            stream_id: PARAMETERS_BY_STREAM[stream_id]
            for stream_id in ("M_fallback", "W_fallback")
        },
        "seasons": fallback_validation_seasons,
    }
)
if FALLBACK_OOF_PATH.exists() and FALLBACK_META_PATH.exists():
    fallback_meta = read_json(FALLBACK_META_PATH)
    assert fallback_meta["signature"] == fallback_signature
    assert fallback_meta["sha256"] == sha256_file(FALLBACK_OOF_PATH)
    fallback_oof = pd.read_parquet(FALLBACK_OOF_PATH)
else:
    fallback_blocks = []
    for stream_id in ("M_fallback", "W_fallback"):
        available = sorted(
            map(int, rows_for_stream(development, stream_id)["Season"].unique())
        )
        for validation_season in fallback_validation_seasons:
            training_seasons = [
                season for season in available if season < validation_season
            ]
            validation = rows_for_stream(
                development,
                stream_id,
                [validation_season],
                output_only=True,
            )
            if not training_seasons or validation.empty:
                continue
            fitted, _ = fit_stream_core(
                development,
                stream_id,
                training_seasons,
                SEED + validation_season,
            )
            block = validation[
                [
                    "TargetKey",
                    "Gender",
                    "Season",
                    "Team1ID",
                    "Team2ID",
                    "Team1Win",
                    "Team1Margin",
                ]
            ].copy()
            block["StreamID"] = stream_id
            block["RawPrediction"] = fitted.predict_raw(validation)
            fallback_blocks.append(block)
            del fitted
            gc.collect()
    fallback_oof = pd.concat(fallback_blocks, ignore_index=True)
    assert set(fallback_oof["StreamID"]) == {"M_fallback", "W_fallback"}
    atomic_parquet(FALLBACK_OOF_PATH, fallback_oof)
    atomic_json(
        FALLBACK_META_PATH,
        {
            "signature": fallback_signature,
            "rows": int(len(fallback_oof)),
            "sha256": sha256_file(FALLBACK_OOF_PATH),
        },
    )


CALIBRATION_BY_STREAM: dict[str, CalibrationSelection] = {}
GUARD_BY_STREAM: dict[str, dict[str, float]] = {}
calibration_reports = []
guard_reports = []
for stream_id in STREAM_SPECS:
    oof = (
        selected_development_oof(stream_id)
        if STREAM_SPECS[stream_id]["seed_aware"]
        else fallback_oof.loc[fallback_oof["StreamID"].eq(stream_id)].copy()
    )
    calibration = select_calibration(oof)
    guard, guard_audit = select_probability_guard(
        oof,
        calibration.cross_fitted,
    )
    CALIBRATION_BY_STREAM[stream_id] = calibration
    GUARD_BY_STREAM[stream_id] = guard
    calibration_reports.append(
        calibration.audit.assign(StreamID=stream_id)
    )
    guard_reports.append(guard_audit.assign(StreamID=stream_id))

atomic_csv(
    REPORT_DIR / "development_calibration_audit.csv",
    pd.concat(calibration_reports, ignore_index=True),
)
atomic_csv(
    REPORT_DIR / "development_probability_guard_audit.csv",
    pd.concat(guard_reports, ignore_index=True),
)


# Learn one primary/challenger blend weight per gender on development OOF only.
def macro_brier_for_weight(
    merged: pd.DataFrame,
    primary_weight: float,
) -> float:
    scored = merged[["Season", "Team1Win"]].copy()
    scored["Prediction"] = (
        primary_weight * merged["PrimaryPrediction"]
        + (1.0 - primary_weight) * merged["ChallengerPrediction"]
    )
    return float(season_brier(scored)["Brier"].mean())


BLEND_WEIGHTS: dict[str, float] = {}
blend_audit_rows = []
for gender in ("M", "W"):
    primary = selected_development_oof(f"{gender}_primary")[
        ["TargetKey", "Season", "Team1Win", "Prediction"]
    ].rename(columns={"Prediction": "PrimaryPrediction"})
    challenger = selected_development_oof(f"{gender}_challenger")[
        ["TargetKey", "Season", "Prediction"]
    ].rename(columns={"Prediction": "ChallengerPrediction"})
    merged = primary.merge(
        challenger,
        on=["TargetKey", "Season"],
        how="inner",
        validate="one_to_one",
    )
    weights = np.linspace(0.0, 1.0, 101)
    scores = np.asarray(
        [macro_brier_for_weight(merged, float(weight)) for weight in weights]
    )
    best_index = int(np.argmin(scores))
    best_weight = float(weights[best_index])
    BLEND_WEIGHTS[gender] = best_weight
    for weight, score in zip(weights, scores, strict=True):
        blend_audit_rows.append(
            {
                "Gender": gender,
                "PrimaryWeight": float(weight),
                "ChallengerWeight": float(1.0 - weight),
                "MacroSeasonBrier": float(score),
                "Selected": bool(float(weight) == best_weight),
            }
        )
blend_audit = pd.DataFrame(blend_audit_rows)
atomic_csv(REPORT_DIR / "development_blend_weight_audit.csv", blend_audit)


FROZEN_BLUEPRINT: dict[str, Any] = {
    "blueprint_version": 3,
    "created_before_locked_labels_loaded": True,
    "source_contracts": EXPECTED_CONTRACTS,
    "locked_seasons": list(LOCKED_SEASONS),
    "target_season": TARGET_SEASON,
    "benchmark_protocols": ["prequential_refit", "static_block"],
    "post_benchmark_retuning_allowed": False,
    "parameter_rule": "median_or_mode_of_matching_nested_development_outer_folds",
    "iteration_rule": "median_inner_selected_iteration_from_development",
    "calibration_rule": "development_only_cross_fitted_one_standard_error_selection",
    "probability_guard_rule": "development_only_shrink_and_clip_without_sharpening",
    "blend_rule": "development_only_macro_season_brier_grid",
    "feature_selector": "notebook03_block_aware_season_stability",
    "stage2_batch_size": 12000,
    "streams": {},
    "blend_weights": BLEND_WEIGHTS,
}
for stream_id, specification in STREAM_SPECS.items():
    FROZEN_BLUEPRINT["streams"][stream_id] = {
        **specification,
        "parameters": PARAMETERS_BY_STREAM[stream_id],
        "fixed_iterations": ITERATIONS_BY_STREAM[stream_id],
        "parameter_provenance": PARAMETER_PROVENANCE[stream_id],
        "calibration_method": CALIBRATION_BY_STREAM[stream_id].method,
        "probability_guard": GUARD_BY_STREAM[stream_id],
    }
FROZEN_BLUEPRINT["blueprint_sha256"] = object_sha256(FROZEN_BLUEPRINT)
BLUEPRINT_PATH = CONFIG_DIR / "benchmark_production.yaml"
if BLUEPRINT_PATH.exists():
    existing_blueprint = yaml.safe_load(BLUEPRINT_PATH.read_text(encoding="utf-8"))
    assert existing_blueprint == FROZEN_BLUEPRINT, (
        "The existing benchmark blueprint differs from the development-frozen "
        "definition. Delete it only if the locked benchmark has never been opened."
    )
else:
    atomic_text(
        BLUEPRINT_PATH,
        yaml.safe_dump(FROZEN_BLUEPRINT, sort_keys=False),
    )
atomic_json(REPORT_DIR / "frozen_benchmark_blueprint.json", FROZEN_BLUEPRINT)

stream_table = pd.DataFrame(
    [
        {
            "StreamID": stream_id,
            "Gender": specification["output_gender"],
            "Role": specification["role"],
            "Architecture": specification["architecture"],
            "Model": specification["model"],
            "ModelDisplay": specification["model_display"],
            "CandidateSet": specification["candidate_set"],
            "Calibration": CALIBRATION_BY_STREAM[stream_id].method,
            "PrimaryWeightInBlend": (
                BLEND_WEIGHTS.get(specification["output_gender"])
                if specification["role"] == "primary"
                else np.nan
            ),
            **GUARD_BY_STREAM[stream_id],
        }
        for stream_id, specification in STREAM_SPECS.items()
    ]
)
atomic_csv(REPORT_DIR / "frozen_stream_definitions.csv", stream_table)

print("Development rows:", f"{len(development):,}")
print("Frozen blueprint:", BLUEPRINT_PATH)
print("Blueprint SHA-256:", FROZEN_BLUEPRINT["blueprint_sha256"])
display(stream_table)


### Failure-focused production preflight

Before the authorization gate can open the locked benchmark, every distinct production stream is fitted on development-only data and asked to score an untouched development season. This verifies package compatibility, feature selection, symmetry, calibration, and the frozen parameter representation without consuming 2022–2025.


In [ ]:
# Development-only execution preflight for every final stream.
preflight_records = []
preflight_validation_season = max(
    map(int, MODEL_CONFIG["matched_outer_validation_seasons"])
)
for stream_id, specification in STREAM_SPECS.items():
    fitted = None
    try:
        available = sorted(
            map(int, rows_for_stream(development, stream_id)["Season"].unique())
        )
        training_seasons = [
            season for season in available if season < preflight_validation_season
        ]
        validation = rows_for_stream(
            development,
            stream_id,
            [preflight_validation_season],
            output_only=True,
        )
        assert training_seasons and not validation.empty
        fitted, selection = fit_stream_core(
            development,
            stream_id,
            training_seasons,
            SEED + preflight_validation_season,
        )
        raw = fitted.predict_raw(validation)
        calibrator = CALIBRATION_BY_STREAM[stream_id].calibrator
        calibrated = symmetric_calibration(calibrator, raw)
        guarded = apply_guard(
            calibrated,
            GUARD_BY_STREAM[stream_id]["shrinkage"],
            GUARD_BY_STREAM[stream_id]["clip_floor"],
        )
        reversed_probability = apply_guard(
            symmetric_calibration(
                calibrator,
                fitted.predict_raw(
                    mirror_features(
                        validation,
                        fitted.features,
                        flip_targets=False,
                    )
                ),
            ),
            GUARD_BY_STREAM[stream_id]["shrinkage"],
            GUARD_BY_STREAM[stream_id]["clip_floor"],
        )
        symmetry_error = float(
            np.max(np.abs(guarded + reversed_probability - 1.0))
        )
        assert np.isfinite(guarded).all()
        assert np.all((guarded >= 0) & (guarded <= 1))
        assert symmetry_error < 1e-6
        preflight_records.append(
            {
                "StreamID": stream_id,
                "Model": specification["model"],
                "Architecture": specification["architecture"],
                "Rows": int(len(validation)),
                "Features": int(len(fitted.features)),
                "SymmetryError": symmetry_error,
                "Status": "complete",
                "Error": None,
            }
        )
    except Exception as exc:
        preflight_records.append(
            {
                "StreamID": stream_id,
                "Model": specification["model"],
                "Architecture": specification["architecture"],
                "Rows": 0,
                "Features": 0,
                "SymmetryError": np.nan,
                "Status": "failed",
                "Error": repr(exc),
            }
        )
    finally:
        try:
            del fitted
        except Exception:
            pass
        gc.collect()

production_preflight = pd.DataFrame(preflight_records)
atomic_csv(REPORT_DIR / "production_preflight.csv", production_preflight)
display(production_preflight)
failed_preflight = production_preflight.loc[
    ~production_preflight["Status"].eq("complete")
]
if not failed_preflight.empty:
    raise AssertionError(
        "PRODUCTION PREFLIGHT FAILED. Review reports/final_2026/"
        "production_preflight.csv. The locked benchmark remains unopened."
    )
print("PRODUCTION PREFLIGHT PASSED.")


## 5. Consume the locked 2022–2025 benchmark once

The first run requires an explicit environment gate. After consumption, immutable cached predictions and a signed lock are reused. A low-memory interruption preserves completed year/stream checkpoints.


In [ ]:
# Explicit benchmark gate and restartable locked evaluation.
BENCHMARK_LOCK_PATH = REPORT_DIR / "benchmark_consumption_lock.json"
BENCHMARK_PREDICTIONS_PATH = PROCESSED / "locked_benchmark_predictions.parquet"
AUTHORIZED = (
    os.environ.get("MM_OPEN_LOCKED_BENCHMARK", "NO").strip().upper()
    == "YES"
)
if BENCHMARK_LOCK_PATH.exists() and not BENCHMARK_PREDICTIONS_PATH.exists():
    raise RuntimeError(
        "The benchmark lock exists but the immutable prediction artifact is missing."
    )
if not BENCHMARK_LOCK_PATH.exists() and not AUTHORIZED:
    raise RuntimeError(
        "LOCKED BENCHMARK NOT OPENED. Save this notebook, stop the complete "
        "Jupyter server, run `set MM_OPEN_LOCKED_BENCHMARK=YES` in Miniforge "
        "Prompt, and relaunch JupyterLab from that prompt. The blueprint is "
        "already frozen on disk."
    )

locked = pd.read_parquet(
    HISTORICAL_PATH,
    columns=history_columns,
    filters=[("DatasetRole", "==", "locked_benchmark")],
)
locked = downcast_numeric(locked)
assert locked["DatasetRole"].eq("locked_benchmark").all()
assert set(map(int, locked["Season"].unique())) == set(LOCKED_SEASONS)
assert locked["TargetKey"].is_unique
assert locked["Team1Win"].isin([0, 1]).all()
assert not development["TargetKey"].isin(locked["TargetKey"]).any()
all_history = pd.concat([development, locked], ignore_index=True)
assert all_history["TargetKey"].is_unique


@dataclass
class StreamBundle:
    stream_id: str
    core: FittedProductionModel
    calibrator_method: str
    calibrator: Any
    guard: dict[str, float]
    training_seasons: list[int]
    selector_hash: str | None

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        raw = self.core.predict_raw(frame)
        calibrated = symmetric_calibration(self.calibrator, raw)
        return apply_guard(
            calibrated,
            self.guard["shrinkage"],
            self.guard["clip_floor"],
        )

    def symmetry_error(self, frame: pd.DataFrame) -> float:
        if frame.empty:
            return 0.0
        forward = self.predict(frame)
        reverse = self.predict(
            mirror_features(
                frame,
                self.core.features,
                flip_targets=False,
            )
        )
        return float(np.max(np.abs(forward + reverse - 1.0)))


def fit_stream_bundle(
    history: pd.DataFrame,
    stream_id: str,
    training_seasons: Sequence[int],
    random_seed: int,
    *,
    calibrator_override: Any | None = None,
) -> tuple[StreamBundle, SelectionResult | None]:
    core, selection = fit_stream_core(
        history,
        stream_id,
        training_seasons,
        random_seed,
    )
    calibration = CALIBRATION_BY_STREAM[stream_id]
    return (
        StreamBundle(
            stream_id=stream_id,
            core=core,
            calibrator_method=calibration.method,
            calibrator=(
                calibrator_override
                if calibrator_override is not None
                else calibration.calibrator
            ),
            guard=GUARD_BY_STREAM[stream_id],
            training_seasons=list(map(int, training_seasons)),
            selector_hash=(
                selection.selector_hash if selection is not None else None
            ),
        ),
        selection,
    )


def score_stream_bundle(
    bundle: StreamBundle,
    validation: pd.DataFrame,
    *,
    protocol: str,
    training_seasons: Sequence[int],
) -> pd.DataFrame:
    specification = STREAM_SPECS[bundle.stream_id]
    base_columns = [
        "TargetKey",
        "GameKey",
        "Gender",
        "Season",
        "DayNum",
        "Team1ID",
        "Team2ID",
        "Team1Win",
        "Team1Margin",
    ]
    diagnostics = [
        column
        for column in (
            "matchup__seed_diff",
            "matchup__seed_gap_abs",
            "matchup__team1_seed",
            "matchup__team2_seed",
        )
        if column in validation.columns
    ]
    output = validation[base_columns + diagnostics].copy()
    raw = bundle.core.predict_raw(validation)
    calibrated = symmetric_calibration(bundle.calibrator, raw)
    prediction = apply_guard(
        calibrated,
        bundle.guard["shrinkage"],
        bundle.guard["clip_floor"],
    )
    output["Protocol"] = protocol
    output["StreamID"] = bundle.stream_id
    output["Role"] = specification["role"]
    output["Architecture"] = specification["architecture"]
    output["Model"] = specification["model"]
    output["ModelDisplay"] = specification["model_display"]
    output["TrainingSeasonMax"] = max(map(int, training_seasons))
    output["FeatureCount"] = len(bundle.core.features)
    output["CalibrationMethod"] = bundle.calibrator_method
    output["RawPrediction"] = raw
    output["CalibratedPrediction"] = calibrated
    output["Prediction"] = prediction
    output["SquaredError"] = (
        prediction - output["Team1Win"].to_numpy(dtype=float)
    ) ** 2
    return output


benchmark_signature = object_sha256(
    {
        "blueprint": FROZEN_BLUEPRINT["blueprint_sha256"],
        "historical_store": EXPECTED_CONTRACTS["historical_store"],
        "locked_seasons": LOCKED_SEASONS,
    }
)
TASK_DIR = CACHE_DIR / "benchmark_tasks" / benchmark_signature[:16]
TASK_DIR.mkdir(parents=True, exist_ok=True)


def run_benchmark_task(
    protocol: str,
    stream_id: str,
    season: int,
    static_bundle: StreamBundle | None,
) -> tuple[pd.DataFrame, StreamBundle | None]:
    stem = f"{protocol}__{stream_id}__{season}"
    prediction_path = TASK_DIR / f"{stem}.parquet"
    metadata_path = TASK_DIR / f"{stem}.json"
    task_signature = object_sha256(
        {
            "benchmark_signature": benchmark_signature,
            "protocol": protocol,
            "stream_id": stream_id,
            "season": int(season),
        }
    )
    if prediction_path.exists() and metadata_path.exists():
        metadata = read_json(metadata_path)
        if (
            metadata.get("task_signature") == task_signature
            and metadata.get("prediction_sha256")
            == sha256_file(prediction_path)
        ):
            return pd.read_parquet(prediction_path), static_bundle

    validation = rows_for_stream(
        locked,
        stream_id,
        [season],
        output_only=True,
    )
    assert not validation.empty
    available = sorted(
        map(int, rows_for_stream(all_history, stream_id)["Season"].unique())
    )
    if protocol == "prequential_refit":
        training_seasons = [value for value in available if value < season]
        bundle, selection = fit_stream_bundle(
            all_history,
            stream_id,
            training_seasons,
            SEED + int(season),
        )
    elif protocol == "static_block":
        training_seasons = [
            value for value in available if value <= DEVELOPMENT_LAST_SEASON
        ]
        if static_bundle is None:
            static_bundle, selection = fit_stream_bundle(
                all_history,
                stream_id,
                training_seasons,
                SEED + DEVELOPMENT_LAST_SEASON,
            )
        else:
            selection = None
        bundle = static_bundle
    else:
        raise KeyError(protocol)

    assert max(training_seasons) < season
    prediction = score_stream_bundle(
        bundle,
        validation,
        protocol=protocol,
        training_seasons=training_seasons,
    )
    atomic_parquet(prediction_path, prediction)
    if selection is not None:
        atomic_csv(
            prediction_path.with_name(
                prediction_path.stem + "__selection.csv"
            ),
            selection.audit,
        )
    atomic_json(
        metadata_path,
        {
            "task_signature": task_signature,
            "protocol": protocol,
            "stream_id": stream_id,
            "season": int(season),
            "training_seasons": training_seasons,
            "features": bundle.core.features,
            "selector_hash": bundle.selector_hash,
            "calibration": bundle.calibrator_method,
            "guard": bundle.guard,
            "parameters": PARAMETERS_BY_STREAM[stream_id],
            "fixed_iterations": ITERATIONS_BY_STREAM[stream_id],
            "prediction_sha256": sha256_file(prediction_path),
        },
    )
    return prediction, static_bundle


if BENCHMARK_PREDICTIONS_PATH.exists() and BENCHMARK_LOCK_PATH.exists():
    benchmark_lock = read_json(BENCHMARK_LOCK_PATH)
    assert benchmark_lock["benchmark_signature"] == benchmark_signature
    assert benchmark_lock["blueprint_sha256"] == FROZEN_BLUEPRINT["blueprint_sha256"]
    assert benchmark_lock["prediction_sha256"] == sha256_file(
        BENCHMARK_PREDICTIONS_PATH
    )
    benchmark_predictions = pd.read_parquet(BENCHMARK_PREDICTIONS_PATH)
    print("Loaded immutable locked-benchmark predictions.")
else:
    benchmark_blocks: list[pd.DataFrame] = []
    static_bundles: dict[str, StreamBundle | None] = {
        stream_id: None
        for stream_id in STREAM_SPECS
        if STREAM_SPECS[stream_id]["role"] in {"primary", "challenger"}
    }
    benchmark_streams = [
        stream_id
        for stream_id in STREAM_SPECS
        if STREAM_SPECS[stream_id]["role"] in {"primary", "challenger"}
    ]
    for protocol in ("prequential_refit", "static_block"):
        for stream_id in benchmark_streams:
            for season in LOCKED_SEASONS:
                print(f"Benchmark: {protocol} | {stream_id} | {season}")
                block, static_bundles[stream_id] = run_benchmark_task(
                    protocol,
                    stream_id,
                    int(season),
                    static_bundles[stream_id],
                )
                benchmark_blocks.append(block)
                gc.collect()
    base_predictions = pd.concat(benchmark_blocks, ignore_index=True)

    blend_blocks = []
    for protocol in ("prequential_refit", "static_block"):
        for gender in ("M", "W"):
            primary = base_predictions.loc[
                base_predictions["Protocol"].eq(protocol)
                & base_predictions["StreamID"].eq(f"{gender}_primary")
            ].copy()
            challenger = base_predictions.loc[
                base_predictions["Protocol"].eq(protocol)
                & base_predictions["StreamID"].eq(f"{gender}_challenger"),
                ["TargetKey", "Season", "Prediction"],
            ].rename(columns={"Prediction": "ChallengerPrediction"})
            merged = primary.merge(
                challenger,
                on=["TargetKey", "Season"],
                how="inner",
                validate="one_to_one",
            )
            weight = float(BLEND_WEIGHTS[gender])
            merged["Prediction"] = clip_probability(
                weight * merged["Prediction"]
                + (1.0 - weight) * merged["ChallengerPrediction"]
            )
            merged["StreamID"] = f"{gender}_blend"
            merged["Role"] = "development_blend"
            merged["Architecture"] = "prediction_blend"
            merged["Model"] = "primary_challenger_blend"
            merged["ModelDisplay"] = "Development-frozen blend"
            merged["CalibrationMethod"] = "component_calibration"
            merged["RawPrediction"] = merged["Prediction"]
            merged["CalibratedPrediction"] = merged["Prediction"]
            merged["SquaredError"] = (
                merged["Prediction"].to_numpy(dtype=float)
                - merged["Team1Win"].to_numpy(dtype=float)
            ) ** 2
            merged = merged.drop(columns=["ChallengerPrediction"])
            blend_blocks.append(merged)

    benchmark_predictions = pd.concat(
        [base_predictions, *blend_blocks],
        ignore_index=True,
    )
    atomic_parquet(BENCHMARK_PREDICTIONS_PATH, benchmark_predictions)

assert benchmark_predictions[
    ["Protocol", "StreamID", "TargetKey"]
].duplicated().sum() == 0
assert set(benchmark_predictions["Protocol"]) == {
    "prequential_refit",
    "static_block",
}
assert set(map(int, benchmark_predictions["Season"].unique())) == set(
    LOCKED_SEASONS
)
assert np.isfinite(benchmark_predictions["Prediction"]).all()
assert benchmark_predictions["Prediction"].between(0, 1).all()

print("LOCKED BENCHMARK PREDICTIONS CREATED.")
print(
    benchmark_predictions.groupby(
        ["Protocol", "Season", "Gender", "Role"],
        observed=True,
    ).size()
)


## 6. Benchmark diagnostics and robustness review

Brier score remains primary because the competition evaluates probability accuracy. The report also includes log loss, ROC AUC, average precision, trapezoidal AUC-PR, accuracy, balanced accuracy, precision, recall, F1, MCC, calibration intercept/slope, expected calibration error, and confusion counts.

Uncertainty comparisons resample whole tournament seasons rather than individual games, preserving the shared-season dependence structure.


In [ ]:
# Locked benchmark metrics, robustness slices, interactive reports, and consumption lock.
def tournament_stage_proxy(day_num: pd.Series) -> pd.Series:
    day = pd.to_numeric(day_num, errors="coerce")
    return pd.Series(
        np.select(
            [
                day.le(135),
                day.between(136, 137),
                day.between(138, 140),
                day.between(141, 144),
                day.between(145, 148),
                day.between(149, 153),
                day.ge(154),
            ],
            [
                "play_in",
                "round_of_64",
                "round_of_32",
                "sweet_16_proxy",
                "elite_8_proxy",
                "final_four_proxy",
                "championship_proxy",
            ],
            default="other",
        ),
        index=day_num.index,
    )


def seed_gap(frame: pd.DataFrame) -> pd.Series:
    for column in ("matchup__seed_gap_abs", "absdiff__seed__seed_number"):
        if column in frame.columns:
            return pd.to_numeric(frame[column], errors="coerce")
    if "matchup__seed_diff" in frame.columns:
        return pd.to_numeric(frame["matchup__seed_diff"], errors="coerce").abs()
    return pd.Series(np.nan, index=frame.index)


def reliability_table(frame: pd.DataFrame, bins: int = 10) -> pd.DataFrame:
    probability = clip_probability(frame["Prediction"])
    target = frame["Team1Win"].to_numpy(dtype=float)
    identifiers = np.clip(
        np.digitize(probability, np.linspace(0, 1, bins + 1), right=True) - 1,
        0,
        bins - 1,
    )
    rows = []
    for index in range(bins):
        mask = identifiers == index
        if mask.any():
            rows.append(
                {
                    "Bin": index,
                    "Rows": int(mask.sum()),
                    "MeanPrediction": float(probability[mask].mean()),
                    "ObservedRate": float(target[mask].mean()),
                }
            )
    return pd.DataFrame(rows)


metric_records = []
season_records = []
for (protocol, stream_id), group in benchmark_predictions.groupby(
    ["Protocol", "StreamID"],
    observed=True,
):
    metadata = group.iloc[0]
    result = probability_metrics(group)
    metric_records.append(
        {
            "Protocol": protocol,
            "StreamID": stream_id,
            "Gender": metadata["Gender"],
            "Role": metadata["Role"],
            "Architecture": metadata["Architecture"],
            "Model": metadata["Model"],
            "ModelDisplay": metadata["ModelDisplay"],
            **result,
        }
    )
    by_season = season_brier(group)
    by_season["Protocol"] = protocol
    by_season["StreamID"] = stream_id
    by_season["Gender"] = metadata["Gender"]
    by_season["Role"] = metadata["Role"]
    by_season["ModelDisplay"] = metadata["ModelDisplay"]
    season_records.append(by_season)

benchmark_metrics = pd.DataFrame(metric_records).sort_values(
    ["Protocol", "Gender", "MacroSeasonBrier", "Role"]
).reset_index(drop=True)
benchmark_by_season = pd.concat(season_records, ignore_index=True)

augmented = benchmark_predictions.copy()
augmented["TournamentStageProxy"] = tournament_stage_proxy(augmented["DayNum"])
augmented["SeedGap"] = seed_gap(augmented)
augmented["SeedGapBand"] = pd.cut(
    augmented["SeedGap"],
    [-np.inf, 0, 2, 5, 9, np.inf],
    labels=["0", "1-2", "3-5", "6-9", "10+"],
)
augmented["Confidence"] = np.maximum(
    augmented["Prediction"],
    1.0 - augmented["Prediction"],
)
augmented["ConfidenceBand"] = pd.cut(
    augmented["Confidence"],
    [0.50, 0.60, 0.70, 0.80, 0.90, 1.00],
    include_lowest=True,
    labels=["0.50-0.60", "0.60-0.70", "0.70-0.80", "0.80-0.90", "0.90-1.00"],
)
augmented["CorrectAt050"] = (
    (augmented["Prediction"] >= 0.5).astype(int)
    == augmented["Team1Win"].astype(int)
)

slice_records = []
for dimension in (
    "TournamentStageProxy",
    "SeedGapBand",
    "ConfidenceBand",
):
    for keys, group in augmented.groupby(
        ["Protocol", "StreamID", "Gender", dimension],
        observed=True,
        dropna=False,
    ):
        if len(group) < 5:
            continue
        protocol, stream_id, gender, level = keys
        metadata = group.iloc[0]
        slice_records.append(
            {
                "Dimension": dimension,
                "Level": str(level),
                "Protocol": protocol,
                "StreamID": stream_id,
                "Gender": gender,
                "Role": metadata["Role"],
                "ModelDisplay": metadata["ModelDisplay"],
                **probability_metrics(group),
            }
        )
benchmark_slices = pd.DataFrame(slice_records)

confident_errors = augmented.loc[
    ((augmented["Prediction"] >= 0.90) & augmented["Team1Win"].eq(0))
    | ((augmented["Prediction"] <= 0.10) & augmented["Team1Win"].eq(1))
].sort_values("SquaredError", ascending=False)


def season_cluster_bootstrap_difference(
    left: pd.DataFrame,
    right: pd.DataFrame,
    *,
    repetitions: int = 5000,
) -> dict[str, float]:
    merged = left[
        ["TargetKey", "Season", "Team1Win", "Prediction"]
    ].rename(columns={"Prediction": "LeftPrediction"}).merge(
        right[["TargetKey", "Season", "Prediction"]].rename(
            columns={"Prediction": "RightPrediction"}
        ),
        on=["TargetKey", "Season"],
        how="inner",
        validate="one_to_one",
    )
    merged["LeftLoss"] = (
        merged["LeftPrediction"] - merged["Team1Win"]
    ) ** 2
    merged["RightLoss"] = (
        merged["RightPrediction"] - merged["Team1Win"]
    ) ** 2
    season_differences = (
        merged.groupby("Season", observed=True)
        .apply(
            lambda group: float(
                group["LeftLoss"].mean() - group["RightLoss"].mean()
            ),
            include_groups=False,
        )
        .to_dict()
    )
    seasons = np.asarray(sorted(season_differences), dtype=int)
    rng = np.random.default_rng(SEED)
    draws = np.empty(repetitions, dtype=float)
    for index in range(repetitions):
        sampled = rng.choice(seasons, size=len(seasons), replace=True)
        draws[index] = float(
            np.mean([season_differences[int(season)] for season in sampled])
        )
    return {
        "ObservedBrierDifferenceLeftMinusRight": float(
            np.mean(list(season_differences.values()))
        ),
        "CILower": float(np.quantile(draws, 0.025)),
        "CIUpper": float(np.quantile(draws, 0.975)),
        "ProbabilityLeftBeatsRight": float(np.mean(draws < 0)),
        "Seasons": int(len(seasons)),
    }


bootstrap_records = []
for protocol in ("prequential_refit", "static_block"):
    for gender in ("M", "W"):
        groups = {
            role: benchmark_predictions.loc[
                benchmark_predictions["Protocol"].eq(protocol)
                & benchmark_predictions["StreamID"].eq(f"{gender}_{role}")
            ]
            for role in ("primary", "challenger", "blend")
        }
        for left_role, right_role in (
            ("primary", "challenger"),
            ("blend", "primary"),
            ("blend", "challenger"),
        ):
            result = season_cluster_bootstrap_difference(
                groups[left_role],
                groups[right_role],
            )
            bootstrap_records.append(
                {
                    "Protocol": protocol,
                    "Gender": gender,
                    "LeftRole": left_role,
                    "RightRole": right_role,
                    **result,
                }
            )
benchmark_bootstrap = pd.DataFrame(bootstrap_records)

atomic_csv(REPORT_DIR / "locked_benchmark_metrics.csv", benchmark_metrics)
atomic_csv(REPORT_DIR / "locked_benchmark_by_season.csv", benchmark_by_season)
atomic_csv(REPORT_DIR / "locked_benchmark_slices.csv", benchmark_slices)
atomic_csv(REPORT_DIR / "locked_benchmark_confident_errors.csv", confident_errors)
atomic_csv(
    REPORT_DIR / "locked_benchmark_season_cluster_bootstrap.csv",
    benchmark_bootstrap,
)

PLOT_ARTIFACTS = list(dict.fromkeys(PLOT_ARTIFACTS))

# 1. Benchmark leaderboard.
plot_frame = benchmark_metrics.loc[
    benchmark_metrics["Protocol"].eq("prequential_refit")
].copy()
figure = px.bar(
    plot_frame.sort_values(["Gender", "MacroSeasonBrier"]),
    x="MacroSeasonBrier",
    y="Role",
    color="ModelDisplay",
    facet_col="Gender",
    orientation="h",
    hover_data=[
        "Architecture",
        "ROCAUC",
        "AveragePrecision",
        "AUCPR",
        "Precision",
        "Recall",
        "F1",
        "CalibrationSlope",
        "ECE",
    ],
    title="Locked benchmark: prequential Brier score",
    labels={"MacroSeasonBrier": "Mean season Brier", "Role": ""},
)
PLOT_ARTIFACTS.append(save_plotly(figure, "locked_benchmark_leaderboard"))

# 2. Season trajectories.
figure = px.line(
    benchmark_by_season.loc[
        benchmark_by_season["Protocol"].eq("prequential_refit")
    ],
    x="Season",
    y="Brier",
    color="ModelDisplay",
    symbol="Role",
    facet_col="Gender",
    markers=True,
    title="Locked benchmark Brier score by tournament season",
)
PLOT_ARTIFACTS.append(save_plotly(figure, "locked_brier_by_season"))

# 3. Protocol sensitivity.
figure = px.bar(
    benchmark_metrics,
    x="Protocol",
    y="MacroSeasonBrier",
    color="Role",
    facet_col="Gender",
    barmode="group",
    hover_data=["ModelDisplay", "ROCAUC", "AveragePrecision", "F1"],
    title="Prequential refit versus static-block sensitivity",
)
PLOT_ARTIFACTS.append(save_plotly(figure, "benchmark_protocol_sensitivity"))

# 4–7. ROC, PR, reliability, and confusion matrices for the primary/challenger/blend streams.
for gender in ("M", "W"):
    streams = augmented.loc[
        augmented["Protocol"].eq("prequential_refit")
        & augmented["Gender"].eq(gender)
    ]
    roc_figure = go.Figure()
    pr_figure = go.Figure()
    reliability_figure = go.Figure()
    confusion_figure = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=("Primary", "Challenger", "Development blend"),
    )
    role_order = ["primary", "challenger", "development_blend"]
    for column_index, role in enumerate(role_order, start=1):
        group = streams.loc[streams["Role"].eq(role)].copy()
        if group.empty:
            continue
        target = group["Team1Win"].to_numpy(dtype=int)
        probability = clip_probability(group["Prediction"])
        fpr, tpr, _ = roc_curve(target, probability)
        precision_values, recall_values, _ = precision_recall_curve(
            target,
            probability,
        )
        label = str(group.iloc[0]["ModelDisplay"])
        roc_figure.add_trace(
            go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{role}: {label}")
        )
        pr_figure.add_trace(
            go.Scatter(
                x=recall_values,
                y=precision_values,
                mode="lines",
                name=f"{role}: {label}",
            )
        )
        reliability = reliability_table(group)
        reliability_figure.add_trace(
            go.Scatter(
                x=reliability["MeanPrediction"],
                y=reliability["ObservedRate"],
                mode="lines+markers",
                text=reliability["Rows"],
                name=f"{role}: {label}",
            )
        )
        matrix = confusion_matrix(
            target,
            (probability >= 0.5).astype(int),
            labels=[0, 1],
        )
        confusion_figure.add_trace(
            go.Heatmap(
                z=matrix,
                x=["Predicted 0", "Predicted 1"],
                y=["Actual 0", "Actual 1"],
                text=matrix,
                texttemplate="%{text}",
                showscale=False,
                name=role,
            ),
            row=1,
            col=column_index,
        )
    roc_figure.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            line={"dash": "dash"},
            name="Chance",
        )
    )
    roc_figure.update_layout(title=f"{gender}: locked benchmark ROC curves")
    pr_figure.update_layout(
        title=f"{gender}: locked benchmark precision-recall curves"
    )
    reliability_figure.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            line={"dash": "dash"},
            name="Perfect calibration",
        )
    )
    reliability_figure.update_layout(
        title=f"{gender}: locked benchmark reliability",
        xaxis_title="Mean predicted probability",
        yaxis_title="Observed win rate",
    )
    confusion_figure.update_layout(
        title=f"{gender}: locked benchmark confusion matrices at 0.50"
    )
    PLOT_ARTIFACTS.append(save_plotly(roc_figure, f"locked_roc_{gender}"))
    PLOT_ARTIFACTS.append(save_plotly(pr_figure, f"locked_pr_{gender}"))
    PLOT_ARTIFACTS.append(
        save_plotly(reliability_figure, f"locked_reliability_{gender}")
    )
    PLOT_ARTIFACTS.append(
        save_plotly(confusion_figure, f"locked_confusion_{gender}")
    )

# Slice report.
if not benchmark_slices.empty:
    figure = px.bar(
        benchmark_slices.loc[
            benchmark_slices["Protocol"].eq("prequential_refit")
            & benchmark_slices["Dimension"].eq("SeedGapBand")
        ],
        x="Level",
        y="GameWeightedBrier",
        color="Role",
        facet_col="Gender",
        barmode="group",
        hover_data=["Rows", "ModelDisplay", "ROCAUC", "F1"],
        title="Locked benchmark Brier score by seed-gap band",
    )
    PLOT_ARTIFACTS.append(save_plotly(figure, "locked_seed_gap_slices"))

new_lock = {
    "lock_version": 2,
    "consumed_at_utc": datetime.now(timezone.utc).isoformat(),
    "benchmark_signature": benchmark_signature,
    "blueprint_sha256": FROZEN_BLUEPRINT["blueprint_sha256"],
    "recipe_sha256": MODEL_RECIPE["recipe_sha256"],
    "historical_store_sha256": EXPECTED_CONTRACTS["historical_store"],
    "locked_seasons": list(LOCKED_SEASONS),
    "prediction_path": str(BENCHMARK_PREDICTIONS_PATH),
    "prediction_sha256": sha256_file(BENCHMARK_PREDICTIONS_PATH),
    "metrics_sha256": sha256_file(REPORT_DIR / "locked_benchmark_metrics.csv"),
    "post_benchmark_retuning_permitted": False,
}
if BENCHMARK_LOCK_PATH.exists():
    benchmark_lock = read_json(BENCHMARK_LOCK_PATH)
    for key in (
        "benchmark_signature",
        "blueprint_sha256",
        "recipe_sha256",
        "historical_store_sha256",
        "prediction_sha256",
    ):
        assert benchmark_lock[key] == new_lock[key], (
            f"Benchmark lock mismatch: {key}"
        )
else:
    benchmark_lock = new_lock
    atomic_json(BENCHMARK_LOCK_PATH, benchmark_lock)

print("LOCKED BENCHMARK CONSUMED AND SEALED.")
display(benchmark_metrics)
display(benchmark_bootstrap)
print("Benchmark lock:", BENCHMARK_LOCK_PATH)


## 7. Final refit through 2025

The frozen procedures are refit using all legally available labeled seasons through 2025. Hyperparameters, feature caps, calibration families, and primary/challenger roles remain unchanged. Only parameter estimates and selected features are refreshed using the enlarged historical sample.


In [ ]:
# Final model refit through 2025 under the unchanged blueprint.
@dataclass
class FinalScoringBundle:
    stream_id: str
    core: FittedProductionModel
    calibrator_method: str
    calibrator: Any
    guard: dict[str, float]
    training_seasons: list[int]
    selector_hash: str | None

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        raw = self.core.predict_raw(frame)
        calibrated = symmetric_calibration(self.calibrator, raw)
        return apply_guard(
            calibrated,
            self.guard["shrinkage"],
            self.guard["clip_floor"],
        )

    def symmetry_error(self, frame: pd.DataFrame) -> float:
        if frame.empty:
            return 0.0
        forward = self.predict(frame)
        reverse = self.predict(
            mirror_features(
                frame,
                self.core.features,
                flip_targets=False,
            )
        )
        return float(np.max(np.abs(forward + reverse - 1.0)))


def final_calibration_rows(stream_id: str) -> pd.DataFrame:
    specification = STREAM_SPECS[stream_id]
    if not specification["seed_aware"]:
        return fallback_oof[[
            "TargetKey",
            "Gender",
            "Season",
            "Team1Win",
            "RawPrediction",
        ]].loc[fallback_oof["StreamID"].eq(stream_id)].copy()

    development_rows = selected_development_oof(stream_id)[
        [
            "TargetKey",
            "Gender",
            "Season",
            "Team1Win",
            "RawPrediction",
        ]
    ].copy()
    benchmark_rows = benchmark_predictions.loc[
        benchmark_predictions["Protocol"].eq("prequential_refit")
        & benchmark_predictions["StreamID"].eq(stream_id),
        [
            "TargetKey",
            "Gender",
            "Season",
            "Team1Win",
            "RawPrediction",
        ],
    ].copy()
    return pd.concat(
        [development_rows, benchmark_rows],
        ignore_index=True,
    ).drop_duplicates(["TargetKey", "Season"], keep="last")


def frame_content_sha256(
    frame: pd.DataFrame,
    columns: Sequence[str],
) -> str:
    ordered = frame[list(columns)].sort_values(
        list(columns[:-1]),
        kind="mergesort",
    ).reset_index(drop=True)
    hashed = pd.util.hash_pandas_object(
        ordered,
        index=False,
        categorize=True,
    ).to_numpy(dtype=np.uint64, copy=True)
    return hashlib.sha256(hashed.tobytes()).hexdigest()


def fitted_feature_importance(
    stream_id: str,
    specification: dict[str, Any],
    core: FittedProductionModel,
) -> pd.DataFrame:
    base = {
        "StreamID": stream_id,
        "Model": specification["model"],
        "Feature": core.features,
        "Block": [
            FEATURE_TO_BLOCK.get(feature, "other")
            for feature in core.features
        ],
    }
    if hasattr(core.model, "coef_"):
        coefficient = np.asarray(core.model.coef_, dtype=float).reshape(-1)
        if len(coefficient) == len(core.features):
            return pd.DataFrame(
                {
                    **base,
                    "ImportanceMethod": "absolute_standardized_coefficient",
                    "Importance": np.abs(coefficient),
                    "SignedValue": coefficient,
                }
            )
    if specification["model"] == "xgb_classifier":
        score = core.model.get_score(importance_type="gain")
        return pd.DataFrame(
            {
                **base,
                "ImportanceMethod": "xgboost_gain",
                "Importance": [
                    float(score.get(feature, 0.0))
                    for feature in core.features
                ],
                "SignedValue": np.nan,
            }
        )
    if specification["model"] == "lgb_classifier":
        return pd.DataFrame(
            {
                **base,
                "ImportanceMethod": "lightgbm_gain",
                "Importance": core.model.feature_importance(
                    importance_type="gain"
                ),
                "SignedValue": np.nan,
            }
        )
    return pd.DataFrame()


FINAL_BUNDLES: dict[str, FinalScoringBundle] = {}
final_model_records: list[dict[str, Any]] = []
final_selection_records: list[pd.DataFrame] = []
final_importance_records: list[pd.DataFrame] = []

for stream_id, specification in STREAM_SPECS.items():
    available = sorted(
        map(int, rows_for_stream(all_history, stream_id)["Season"].unique())
    )
    training_seasons = [
        season for season in available if season <= max(LOCKED_SEASONS)
    ]
    calibration_rows = final_calibration_rows(stream_id)
    calibration_method = CALIBRATION_BY_STREAM[stream_id].method
    calibration_hash = frame_content_sha256(
        calibration_rows,
        ["Season", "TargetKey", "Team1Win", "RawPrediction"],
    )
    refit_signature = object_sha256(
        {
            "blueprint_sha256": FROZEN_BLUEPRINT["blueprint_sha256"],
            "recipe_sha256": MODEL_RECIPE["recipe_sha256"],
            "historical_store_sha256": EXPECTED_CONTRACTS[
                "historical_store"
            ],
            "benchmark_prediction_sha256": benchmark_lock[
                "prediction_sha256"
            ],
            "stream": specification,
            "training_seasons": training_seasons,
            "parameters": PARAMETERS_BY_STREAM[stream_id],
            "fixed_iterations": ITERATIONS_BY_STREAM[stream_id],
            "calibration_method": calibration_method,
            "calibration_rows_sha256": calibration_hash,
            "probability_guard": GUARD_BY_STREAM[stream_id],
        }
    )

    stream_directory = MODEL_DIR / stream_id
    stream_directory.mkdir(parents=True, exist_ok=True)
    model_path = stream_directory / "scoring_bundle.joblib"
    metadata_path = stream_directory / "metadata.json"
    selection_path = stream_directory / "feature_selection.csv"
    importance_path = stream_directory / "feature_importance.csv"

    loaded_from_checkpoint = False
    if model_path.exists() and metadata_path.exists():
        cached_metadata = read_json(metadata_path)
        if (
            cached_metadata.get("final_refit_signature") == refit_signature
            and cached_metadata.get("model_file_sha256")
            == sha256_file(model_path)
        ):
            try:
                bundle = joblib.load(model_path)
                assert bundle.stream_id == stream_id
                assert bundle.training_seasons == training_seasons
                FINAL_BUNDLES[stream_id] = bundle
                loaded_from_checkpoint = True
                if selection_path.exists():
                    final_selection_records.append(
                        pd.read_csv(selection_path)
                    )
                if importance_path.exists():
                    cached_importance = pd.read_csv(importance_path)
                    if not cached_importance.empty:
                        final_importance_records.append(cached_importance)
                record = dict(cached_metadata)
                record["loaded_from_checkpoint"] = True
                final_model_records.append(record)
                print(
                    "Final refit loaded from checkpoint:",
                    stream_id,
                    specification["model"],
                )
            except Exception as exc:
                print(
                    "Final-refit checkpoint could not be loaded; refitting",
                    stream_id,
                    repr(exc),
                )

    if loaded_from_checkpoint:
        gc.collect()
        continue

    print("Final refit:", stream_id, specification["model"])
    core, selection = fit_stream_core(
        all_history,
        stream_id,
        training_seasons,
        SEED + TARGET_SEASON,
    )
    final_calibrator = fit_calibrator(
        calibration_method,
        calibration_rows["RawPrediction"].to_numpy(dtype=float),
        calibration_rows["Team1Win"].to_numpy(dtype=int),
    )
    bundle = FinalScoringBundle(
        stream_id=stream_id,
        core=core,
        calibrator_method=calibration_method,
        calibrator=final_calibrator,
        guard=GUARD_BY_STREAM[stream_id],
        training_seasons=training_seasons,
        selector_hash=(
            selection.selector_hash if selection is not None else None
        ),
    )
    FINAL_BUNDLES[stream_id] = bundle

    if selection is not None:
        selection_table = selection.audit.copy()
        selection_table["StreamID"] = stream_id
        final_selection_records.append(selection_table)
        atomic_csv(selection_path, selection_table)

    importance = fitted_feature_importance(
        stream_id,
        specification,
        core,
    )
    if not importance.empty:
        final_importance_records.append(importance)
        atomic_csv(importance_path, importance)

    atomic_joblib(model_path, bundle)
    metadata = {
        "stream_id": stream_id,
        "output_gender": specification["output_gender"],
        "role": specification["role"],
        "architecture": specification["architecture"],
        "training_gender": specification["training_gender"],
        "model": specification["model"],
        "model_display": specification["model_display"],
        "seed_aware": specification["seed_aware"],
        "candidate_set": specification["candidate_set"],
        "training_seasons": training_seasons,
        "selected_features": core.features,
        "feature_count": len(core.features),
        "selector_hash": bundle.selector_hash,
        "parameters": PARAMETERS_BY_STREAM[stream_id],
        "fixed_iterations": ITERATIONS_BY_STREAM[stream_id],
        "calibration_method": calibration_method,
        "calibration_rows": int(len(calibration_rows)),
        "calibration_rows_sha256": calibration_hash,
        "probability_guard": bundle.guard,
        "blueprint_sha256": FROZEN_BLUEPRINT["blueprint_sha256"],
        "recipe_sha256": MODEL_RECIPE["recipe_sha256"],
        "final_refit_signature": refit_signature,
        "model_file": str(model_path),
        "model_file_sha256": sha256_file(model_path),
        "loaded_from_checkpoint": False,
    }
    atomic_json(metadata_path, metadata)
    final_model_records.append(metadata)
    gc.collect()

final_model_manifest = pd.DataFrame(final_model_records)
atomic_csv(REPORT_DIR / "final_model_manifest.csv", final_model_manifest)

final_feature_selection = (
    pd.concat(final_selection_records, ignore_index=True)
    if final_selection_records
    else pd.DataFrame()
)
atomic_csv(REPORT_DIR / "final_feature_selection.csv", final_feature_selection)

final_feature_importance = (
    pd.concat(final_importance_records, ignore_index=True)
    if final_importance_records
    else pd.DataFrame()
)
if not final_feature_importance.empty:
    final_feature_importance = final_feature_importance.sort_values(
        ["StreamID", "Importance"],
        ascending=[True, False],
    )
atomic_csv(REPORT_DIR / "final_feature_importance.csv", final_feature_importance)

symmetry_records = []
for stream_id, bundle in FINAL_BUNDLES.items():
    sample = rows_for_stream(
        all_history,
        stream_id,
        [max(LOCKED_SEASONS)],
        output_only=True,
    ).head(50)
    symmetry_records.append(
        {
            "StreamID": stream_id,
            "Rows": int(len(sample)),
            "MaximumSymmetryError": bundle.symmetry_error(sample),
        }
    )
final_model_symmetry = pd.DataFrame(symmetry_records)
atomic_csv(REPORT_DIR / "final_model_symmetry_audit.csv", final_model_symmetry)
assert final_model_symmetry["MaximumSymmetryError"].max() < 1e-6

if not final_feature_importance.empty:
    plot_importance = (
        final_feature_importance.sort_values(
            ["StreamID", "Importance"],
            ascending=[True, False],
        )
        .groupby("StreamID", observed=True)
        .head(15)
    )
    figure = px.bar(
        plot_importance,
        x="Importance",
        y="Feature",
        color="Block",
        facet_col="StreamID",
        facet_col_wrap=2,
        orientation="h",
        hover_data=["Model", "ImportanceMethod", "SignedValue"],
        title="Final fitted-model feature importance",
        height=max(
            750,
            340 * math.ceil(plot_importance["StreamID"].nunique() / 2),
        ),
    )
    figure.update_yaxes(matches=None, showticklabels=True)
    PLOT_ARTIFACTS.append(
        save_plotly(figure, "final_model_feature_importance")
    )

print("Final refits complete:", sorted(FINAL_BUNDLES))
display(
    final_model_manifest[
        [
            "stream_id",
            "role",
            "architecture",
            "model_display",
            "feature_count",
            "calibration_method",
            "calibration_rows",
            "loaded_from_checkpoint",
        ]
    ]
)


## 8. Projected, restartable Stage 2 scoring

The notebook first obtains the final selected-feature union, then reads only those columns from the wide Stage 2 Parquet file. Scoring occurs in bounded batches with per-batch hashes, so an interrupted run resumes from verified chunks.

Rows with both seeds use the primary/challenger seed-aware streams. Hypothetical pairs lacking one or both seeds use the documented seed-free fallback. Actual tournament games should use the seed-aware path because Selection Sunday assigns a seed to every selected team.


In [ ]:
# Projected and restartable Stage 2 scoring.
assert sha256_file(STAGE2_PATH) == EXPECTED_CONTRACTS["stage2_store"], (
    "The Stage 2 feature store changed after notebook 02."
)
stage2_schema = set(pq.ParquetFile(STAGE2_PATH).schema.names)
required_features = sorted(
    set().union(
        *(set(bundle.core.features) for bundle in FINAL_BUNDLES.values())
    )
)
missing_features = sorted(set(required_features).difference(stage2_schema))
assert not missing_features, (
    "Stage 2 is missing final selected features: "
    f"{missing_features[:20]}"
)
STAGE2_METADATA = [
    "ID",
    "Gender",
    "Season",
    "Team1ID",
    "Team2ID",
    "FeatureRoute",
    "RequestOrderInternal",
    "matchup__both_seeds_available",
]
stage2_columns = [
    column
    for column in STAGE2_METADATA + required_features
    if column in stage2_schema
]
for required in ("ID", "Gender", "Season", "Team1ID", "Team2ID"):
    assert required in stage2_columns

scoring_signature = object_sha256(
    {
        "blueprint": FROZEN_BLUEPRINT["blueprint_sha256"],
        "stage2_store": EXPECTED_CONTRACTS["stage2_store"],
        "model_files": {
            row["stream_id"]: row["model_file_sha256"]
            for _, row in final_model_manifest.iterrows()
        },
        "columns": stage2_columns,
        "batch_size": int(FROZEN_BLUEPRINT["stage2_batch_size"]),
    }
)
chunk_directory = CACHE_DIR / "stage2_chunks" / scoring_signature[:16]
chunk_directory.mkdir(parents=True, exist_ok=True)
batch_size = int(FROZEN_BLUEPRINT["stage2_batch_size"])


def stage2_seed_aware_mask(frame: pd.DataFrame) -> pd.Series:
    if "FeatureRoute" in frame.columns:
        return frame["FeatureRoute"].astype(str).eq("seed_aware")
    if "matchup__both_seeds_available" in frame.columns:
        return frame["matchup__both_seeds_available"].fillna(False).astype(bool)
    raise KeyError(
        "Stage 2 requires FeatureRoute or matchup__both_seeds_available."
    )


def score_stage2_batch(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame[
        ["ID", "Gender", "Season", "Team1ID", "Team2ID"]
    ].copy()
    seed_aware = stage2_seed_aware_mask(frame)
    output["FeatureRoute"] = np.where(
        seed_aware,
        "seed_aware",
        "seed_free",
    )
    output["PredPrimary"] = np.nan
    output["PredChallenger"] = np.nan
    output["PredBlend"] = np.nan
    output["PrimaryRoute"] = ""
    output["ChallengerRoute"] = ""

    for gender in ("M", "W"):
        gender_mask = frame["Gender"].eq(gender)
        aware_mask = gender_mask & seed_aware
        fallback_mask = gender_mask & ~seed_aware

        if aware_mask.any():
            primary_id = f"{gender}_primary"
            challenger_id = f"{gender}_challenger"
            primary_probability = FINAL_BUNDLES[primary_id].predict(
                frame.loc[aware_mask]
            )
            challenger_probability = FINAL_BUNDLES[challenger_id].predict(
                frame.loc[aware_mask]
            )
            weight = float(BLEND_WEIGHTS[gender])
            blend_probability = clip_probability(
                weight * primary_probability
                + (1.0 - weight) * challenger_probability
            )
            output.loc[aware_mask, "PredPrimary"] = primary_probability
            output.loc[aware_mask, "PredChallenger"] = challenger_probability
            output.loc[aware_mask, "PredBlend"] = blend_probability
            output.loc[aware_mask, "PrimaryRoute"] = primary_id
            output.loc[aware_mask, "ChallengerRoute"] = challenger_id

        if fallback_mask.any():
            fallback_id = f"{gender}_fallback"
            fallback_probability = FINAL_BUNDLES[fallback_id].predict(
                frame.loc[fallback_mask]
            )
            for column in ("PredPrimary", "PredChallenger", "PredBlend"):
                output.loc[fallback_mask, column] = fallback_probability
            output.loc[fallback_mask, "PrimaryRoute"] = fallback_id
            output.loc[fallback_mask, "ChallengerRoute"] = fallback_id

    return output


scored_blocks = []
row_offset = 0
parquet_file = pq.ParquetFile(STAGE2_PATH)
for batch_number, batch in enumerate(
    parquet_file.iter_batches(
        batch_size=batch_size,
        columns=stage2_columns,
        use_threads=False,
    )
):
    batch_frame = batch.to_pandas()
    batch_frame["RowNumber"] = np.arange(
        row_offset,
        row_offset + len(batch_frame),
        dtype=np.int64,
    )
    row_offset += len(batch_frame)
    chunk_path = chunk_directory / f"chunk_{batch_number:05d}.parquet"
    metadata_path = chunk_directory / f"chunk_{batch_number:05d}.json"
    chunk_signature = object_sha256(
        {
            "scoring_signature": scoring_signature,
            "batch_number": batch_number,
            "row_start": int(batch_frame["RowNumber"].min()),
            "row_end": int(batch_frame["RowNumber"].max()),
            "ids_hash": object_sha256(batch_frame["ID"].astype(str).tolist()),
        }
    )
    if chunk_path.exists() and metadata_path.exists():
        metadata = read_json(metadata_path)
        if (
            metadata.get("chunk_signature") == chunk_signature
            and metadata.get("sha256") == sha256_file(chunk_path)
        ):
            scored_blocks.append(pd.read_parquet(chunk_path))
            continue

    print(
        f"Stage 2 batch {batch_number + 1}: rows "
        f"{int(batch_frame['RowNumber'].min()):,}-"
        f"{int(batch_frame['RowNumber'].max()):,}"
    )
    ensure_memory(
        f"Stage 2 batch {batch_number + 1}",
        minimum_available_gb=0.40,
    )
    scored = score_stage2_batch(batch_frame)
    scored["RowNumber"] = batch_frame["RowNumber"].to_numpy()
    assert scored[["PredPrimary", "PredChallenger", "PredBlend"]].notna().all().all()
    for column in ("PredPrimary", "PredChallenger", "PredBlend"):
        assert np.isfinite(scored[column]).all()
        assert scored[column].between(0, 1).all()
    atomic_parquet(chunk_path, scored)
    atomic_json(
        metadata_path,
        {
            "chunk_signature": chunk_signature,
            "rows": int(len(scored)),
            "sha256": sha256_file(chunk_path),
        },
    )
    scored_blocks.append(scored)
    del batch_frame, scored
    gc.collect()

stage2_scored = (
    pd.concat(scored_blocks, ignore_index=True)
    .sort_values("RowNumber")
    .reset_index(drop=True)
)
stage2_ids = pd.read_parquet(STAGE2_PATH, columns=["ID"])["ID"].astype(str).reset_index(drop=True)
assert stage2_scored["ID"].astype(str).reset_index(drop=True).equals(stage2_ids)
assert len(stage2_scored) == int(READINESS_02["stage2_matchup_rows"])
assert stage2_scored["ID"].is_unique
assert stage2_scored["Season"].eq(TARGET_SEASON).all()
assert stage2_scored["Team1ID"].lt(stage2_scored["Team2ID"]).all()

# Cross-check the routing artifact from notebook 01 when present.
if SUBMISSION_ROUTING_PATH.exists():
    routing = pd.read_parquet(SUBMISSION_ROUTING_PATH)
    if "Season" in routing.columns:
        routing = routing.loc[routing["Season"].eq(TARGET_SEASON)].copy()
    if "RequestOrderInternal" in routing.columns:
        routing = routing.sort_values("RequestOrderInternal")
    routing_ids = routing["ID"].astype(str).reset_index(drop=True)
    assert routing_ids.equals(stage2_ids)

STAGE2_PREDICTIONS_PATH = PROCESSED / "stage2_predictions_2026.parquet"
atomic_parquet(STAGE2_PREDICTIONS_PATH, stage2_scored)

submission_primary = stage2_scored[["ID", "PredPrimary"]].rename(
    columns={"PredPrimary": "Pred"}
)
submission_challenger = stage2_scored[["ID", "PredChallenger"]].rename(
    columns={"PredChallenger": "Pred"}
)
submission_blend = stage2_scored[["ID", "PredBlend"]].rename(
    columns={"PredBlend": "Pred"}
)

SUBMISSION_PRIMARY_PATH = SUBMISSION_DIR / "submission_2026_primary.csv"
SUBMISSION_CHALLENGER_PATH = SUBMISSION_DIR / "submission_2026_challenger.csv"
SUBMISSION_BLEND_PATH = SUBMISSION_DIR / "submission_2026_development_blend.csv"
SUBMISSION_DEFAULT_PATH = SUBMISSION_DIR / "submission_2026.csv"
atomic_csv(SUBMISSION_PRIMARY_PATH, submission_primary)
atomic_csv(SUBMISSION_CHALLENGER_PATH, submission_challenger)
atomic_csv(SUBMISSION_BLEND_PATH, submission_blend)
atomic_csv(SUBMISSION_DEFAULT_PATH, submission_primary)

route_summary = (
    stage2_scored.groupby(
        ["Gender", "FeatureRoute", "PrimaryRoute", "ChallengerRoute"],
        observed=True,
    )
    .size()
    .rename("Rows")
    .reset_index()
)
atomic_csv(REPORT_DIR / "stage2_route_summary.csv", route_summary)

submission_manifest = pd.DataFrame(
    [
        {
            "Submission": label,
            "Path": str(path),
            "Rows": int(len(frame)),
            "SHA256": sha256_file(path),
            "Purpose": purpose,
        }
        for label, path, frame, purpose in (
            (
                "primary",
                SUBMISSION_PRIMARY_PATH,
                submission_primary,
                "Predeclared one-standard-error recommendation",
            ),
            (
                "challenger",
                SUBMISSION_CHALLENGER_PATH,
                submission_challenger,
                "Lowest development Brier-score model",
            ),
            (
                "development_blend",
                SUBMISSION_BLEND_PATH,
                submission_blend,
                "Blend weight frozen on development OOF",
            ),
            (
                "default",
                SUBMISSION_DEFAULT_PATH,
                submission_primary,
                "Alias of the primary submission",
            ),
        )
    ]
)
atomic_csv(REPORT_DIR / "submission_manifest.csv", submission_manifest)

prediction_long = stage2_scored.melt(
    id_vars=["Gender", "FeatureRoute"],
    value_vars=["PredPrimary", "PredChallenger", "PredBlend"],
    var_name="Submission",
    value_name="Probability",
)
figure = px.histogram(
    prediction_long,
    x="Probability",
    color="Submission",
    facet_row="Gender",
    facet_col="FeatureRoute",
    nbins=40,
    barmode="overlay",
    opacity=0.55,
    title="2026 Stage 2 probability distributions",
)
PLOT_ARTIFACTS.append(save_plotly(figure, "stage2_probability_distributions"))

figure = px.scatter(
    stage2_scored.sample(
        n=min(10000, len(stage2_scored)),
        random_state=SEED,
    ),
    x="PredPrimary",
    y="PredChallenger",
    color="Gender",
    symbol="FeatureRoute",
    opacity=0.45,
    title="Primary versus challenger 2026 probabilities",
)
figure.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=1,
    y1=1,
    line={"dash": "dash"},
)
PLOT_ARTIFACTS.append(save_plotly(figure, "stage2_primary_vs_challenger"))

print("Stage 2 scoring complete:", f"{len(stage2_scored):,}")
display(route_summary)
display(submission_manifest)


## 9. Exact bracket propagation

Where 2026 seeds and slot definitions are present, advancement probabilities are propagated exactly through the bracket graph. This is deterministic dynamic programming, not Monte Carlo simulation, so the reported round and championship probabilities contain no simulation noise.


In [ ]:
# Exact probabilistic bracket propagation for the primary submission.
SEEDS_PATH = INTERIM / "seeds.parquet"
TEAMS_PATH = INTERIM / "teams.parquet"


def first_existing_path(candidates: Sequence[Path]) -> Path | None:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def first_raw_csv(filename: str) -> Path | None:
    matches = sorted((ROOT / "data" / "raw").glob(f"**/{filename}"))
    return matches[0] if matches else None


def read_reference_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported reference-table format: {path}")


SLOT_PATHS = {
    "M": first_existing_path(
        [
            INTERIM / "men_tourney_slots.parquet",
            INTERIM / "MNCAATourneySlots.parquet",
        ]
        + ([first_raw_csv("MNCAATourneySlots.csv")] if first_raw_csv("MNCAATourneySlots.csv") is not None else [])
    ),
    "W": first_existing_path(
        [
            INTERIM / "women_tourney_slots.parquet",
            INTERIM / "WNCAATourneySlots.parquet",
        ]
        + ([first_raw_csv("WNCAATourneySlots.csv")] if first_raw_csv("WNCAATourneySlots.csv") is not None else [])
    ),
}
BRACKET_AVAILABLE = (
    SEEDS_PATH.exists()
    and TEAMS_PATH.exists()
    and all(path is not None and path.exists() for path in SLOT_PATHS.values())
)


def pair_probability(
    lookup: dict[tuple[str, int, int], float],
    gender: str,
    team_a: int,
    team_b: int,
) -> float:
    if team_a == team_b:
        return 0.5
    lower, upper = sorted((int(team_a), int(team_b)))
    key = (str(gender), lower, upper)
    if key not in lookup:
        raise KeyError(f"Stage 2 probability is missing for {key}.")
    probability_lower = float(lookup[key])
    return probability_lower if int(team_a) == lower else 1.0 - probability_lower


def slot_round(slot: str) -> int:
    match = __import__("re").match(r"R(\d+)", str(slot))
    return int(match.group(1)) if match else 0


def propagate_bracket(
    *,
    gender: str,
    slots: pd.DataFrame,
    seeds: pd.DataFrame,
    teams: pd.DataFrame,
    lookup: dict[tuple[str, int, int], float],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    season_slots = slots.loc[slots["Season"].eq(TARGET_SEASON)].copy()
    season_seeds = seeds.loc[
        seeds["Season"].eq(TARGET_SEASON)
        & seeds["Gender"].eq(gender)
    ].copy()
    if season_slots.empty or season_seeds.empty:
        raise ValueError(
            f"Missing {gender} bracket slots or seeds for {TARGET_SEASON}."
        )
    assert {"Slot", "StrongSeed", "WeakSeed"}.issubset(season_slots.columns)
    assert {"Seed", "TeamID"}.issubset(season_seeds.columns)

    team_name = dict(
        zip(
            teams.loc[teams["Gender"].eq(gender), "TeamID"].astype(int),
            teams.loc[teams["Gender"].eq(gender), "TeamName"].astype(str),
            strict=True,
        )
    )
    distributions: dict[str, dict[int, float]] = {
        str(row.Seed): {int(row.TeamID): 1.0}
        for row in season_seeds[["Seed", "TeamID"]].itertuples(index=False)
    }
    unresolved = season_slots[["Slot", "StrongSeed", "WeakSeed"]].copy()
    slot_records: list[dict[str, Any]] = []
    audit_records: list[dict[str, Any]] = []

    while not unresolved.empty:
        progress = False
        keep_indices = []
        for index, row in unresolved.iterrows():
            slot = str(row["Slot"])
            strong = str(row["StrongSeed"])
            weak = str(row["WeakSeed"])
            if strong not in distributions or weak not in distributions:
                keep_indices.append(index)
                continue
            winner_distribution: dict[int, float] = {}
            for team_a, path_a in distributions[strong].items():
                for team_b, path_b in distributions[weak].items():
                    path_probability = float(path_a) * float(path_b)
                    if path_probability <= 0:
                        continue
                    p_a = pair_probability(
                        lookup,
                        gender,
                        team_a,
                        team_b,
                    )
                    winner_distribution[team_a] = (
                        winner_distribution.get(team_a, 0.0)
                        + path_probability * p_a
                    )
                    winner_distribution[team_b] = (
                        winner_distribution.get(team_b, 0.0)
                        + path_probability * (1.0 - p_a)
                    )
            total = float(sum(winner_distribution.values()))
            if not np.isfinite(total) or abs(total - 1.0) > 1e-10:
                raise AssertionError(
                    f"{gender}/{slot} probabilities sum to {total}."
                )
            distributions[slot] = winner_distribution
            round_number = slot_round(slot)
            for team_id, probability in winner_distribution.items():
                slot_records.append(
                    {
                        "Gender": gender,
                        "Season": TARGET_SEASON,
                        "Slot": slot,
                        "Round": round_number,
                        "TeamID": int(team_id),
                        "TeamName": team_name.get(int(team_id), str(team_id)),
                        "WinSlotProbability": float(probability),
                    }
                )
            audit_records.append(
                {
                    "Gender": gender,
                    "Slot": slot,
                    "Round": round_number,
                    "StrongSource": strong,
                    "WeakSource": weak,
                    "TeamsInDistribution": len(winner_distribution),
                    "ProbabilitySum": total,
                    "Passed": abs(total - 1.0) <= 1e-10,
                }
            )
            progress = True
        if not progress:
            unresolved_tokens = unresolved.loc[keep_indices].to_dict(
                orient="records"
            )
            raise RuntimeError(
                f"Could not resolve {gender} bracket slots: "
                f"{unresolved_tokens[:10]}"
            )
        unresolved = unresolved.loc[keep_indices].copy()

    slot_table = pd.DataFrame(slot_records)
    audit = pd.DataFrame(audit_records)
    final_round = int(slot_table["Round"].max())
    final_slots = sorted(
        audit.loc[audit["Round"].eq(final_round), "Slot"].tolist()
    )
    championship_slot = final_slots[-1]
    champion = (
        slot_table.loc[slot_table["Slot"].eq(championship_slot)]
        .rename(columns={"WinSlotProbability": "ChampionshipProbability"})
        .sort_values("ChampionshipProbability", ascending=False)
        .reset_index(drop=True)
    )
    assert abs(float(champion["ChampionshipProbability"].sum()) - 1.0) <= 1e-10
    return slot_table, champion, audit


bracket_slot_probabilities = pd.DataFrame()
championship_probabilities = pd.DataFrame()
bracket_audit = pd.DataFrame()
if BRACKET_AVAILABLE:
    seeds_2026 = pd.read_parquet(SEEDS_PATH)
    teams_reference = pd.read_parquet(TEAMS_PATH)
    probability_lookup = {
        (str(row.Gender), int(row.Team1ID), int(row.Team2ID)): float(
            row.PredPrimary
        )
        for row in stage2_scored[
            ["Gender", "Team1ID", "Team2ID", "PredPrimary"]
        ].itertuples(index=False)
    }
    slot_blocks, champion_blocks, audit_blocks = [], [], []
    for gender, path in SLOT_PATHS.items():
        assert path is not None
        slots = read_reference_table(path)
        slot_table, champion, audit = propagate_bracket(
            gender=gender,
            slots=slots,
            seeds=seeds_2026,
            teams=teams_reference,
            lookup=probability_lookup,
        )
        slot_blocks.append(slot_table)
        champion_blocks.append(champion)
        audit_blocks.append(audit)
    bracket_slot_probabilities = pd.concat(slot_blocks, ignore_index=True)
    championship_probabilities = pd.concat(champion_blocks, ignore_index=True)
    bracket_audit = pd.concat(audit_blocks, ignore_index=True)
    assert bracket_audit["Passed"].all()
    atomic_csv(
        REPORT_DIR / "bracket_slot_probabilities.csv",
        bracket_slot_probabilities,
    )
    atomic_csv(
        REPORT_DIR / "championship_probabilities.csv",
        championship_probabilities,
    )
    atomic_csv(
        REPORT_DIR / "bracket_probability_audit.csv",
        bracket_audit,
    )

    for gender in ("M", "W"):
        top = championship_probabilities.loc[
            championship_probabilities["Gender"].eq(gender)
        ].head(20)
        figure = px.bar(
            top.sort_values("ChampionshipProbability"),
            x="ChampionshipProbability",
            y="TeamName",
            orientation="h",
            hover_data=["TeamID", "Slot"],
            title=(
                f"{TARGET_SEASON} "
                f"{'men' if gender == 'M' else 'women'}: "
                "primary-model championship probabilities"
            ),
        )
        PLOT_ARTIFACTS.append(
            save_plotly(figure, f"championship_probabilities_{gender}")
        )
    print("Exact bracket propagation complete.")
else:
    print(
        "Optional bracket reporting skipped because a complete 2026 "
        "seed/slot reference set was not found. Submission scoring and "
        "all mandatory audits remain complete."
    )


## 10. Final audit, model card, and reproducibility manifest

Completion is blocked unless the benchmark lock, final models, scoring chunks, submission schema, row order, probability bounds, route coverage, symmetry checks, artifact hashes, and report inventory all pass.


In [ ]:
# Final integrity gate, model card, and reproducibility manifest.
FINAL_CHECKS: list[dict[str, Any]] = []


def add_final_check(
    name: str,
    passed: bool,
    details: str,
    *,
    blocking: bool = True,
) -> None:
    FINAL_CHECKS.append(
        {
            "Check": name,
            "Passed": bool(passed),
            "Blocking": bool(blocking),
            "Details": details,
        }
    )


add_final_check(
    "notebook 03 self-contained integrity gate passed",
    (
        READINESS_03["status"] == "complete"
        and READINESS_03["blocking_final_check_failures"] == 0
        and FINALIZATION_03["status"] == "complete"
        and int(FINALIZATION_03.get("metric_integrity_issues", 0)) == 0
        and METRIC_ISSUES_03.empty
    ),
    f"recipe={MODEL_RECIPE['recipe_sha256']}",
)
add_final_check(
    "notebook 03 metric integrity report is empty",
    METRIC_ISSUES_03.empty,
    f"rows={len(METRIC_ISSUES_03)}",
)
add_final_check(
    "notebook 03 framework coverage is complete",
    as_bool_series(FRAMEWORK_COVERAGE_03["Complete"]).all(),
    FRAMEWORK_COVERAGE_03[
        ["Gender", "Architecture", "Complete", "MissingModels"]
    ].to_json(orient="records"),
)
add_final_check(
    "development handoff reports were written before benchmark access",
    (
        (REPORT_DIR / "development_handoff_decisions.csv").exists()
        and (REPORT_DIR / "development_neural_model_audit.csv").exists()
    ),
    "primary, challenger, and neural audit",
)

add_final_check(
    "benchmark blueprint is unchanged",
    yaml.safe_load(BLUEPRINT_PATH.read_text(encoding="utf-8"))
    == FROZEN_BLUEPRINT,
    FROZEN_BLUEPRINT["blueprint_sha256"],
)
add_final_check(
    "locked benchmark consumption lock is present",
    BENCHMARK_LOCK_PATH.exists(),
    str(BENCHMARK_LOCK_PATH),
)
add_final_check(
    "locked benchmark predictions match the lock",
    (
        benchmark_lock["prediction_sha256"]
        == sha256_file(BENCHMARK_PREDICTIONS_PATH)
        and benchmark_lock["blueprint_sha256"]
        == FROZEN_BLUEPRINT["blueprint_sha256"]
        and benchmark_lock["post_benchmark_retuning_permitted"] is False
    ),
    json.dumps(benchmark_lock, default=str),
)
add_final_check(
    "every locked game has primary, challenger, and blend predictions",
    (
        benchmark_predictions.groupby(
            ["Protocol", "Gender", "TargetKey"],
            observed=True,
        )["Role"].nunique().eq(3).all()
    ),
    benchmark_predictions.groupby(
        ["Protocol", "Gender"],
        observed=True,
    )["Role"].nunique().to_dict().__repr__(),
)
add_final_check(
    "benchmark metric suite is finite",
    benchmark_metrics[
        [
            "MacroSeasonBrier",
            "GameWeightedBrier",
            "LogLoss",
            "ROCAUC",
            "AveragePrecision",
            "AUCPR",
            "Accuracy",
            "BalancedAccuracy",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "ECE",
        ]
    ].replace([np.inf, -np.inf], np.nan).notna().all().all(),
    f"rows={len(benchmark_metrics)}",
)
add_final_check(
    "all six final scoring bundles were fitted",
    set(FINAL_BUNDLES) == set(STREAM_SPECS),
    str(sorted(FINAL_BUNDLES)),
)
add_final_check(
    "final fitted models preserve probability symmetry",
    final_model_symmetry["MaximumSymmetryError"].max() < 1e-6,
    f"maximum={final_model_symmetry['MaximumSymmetryError'].max():.3e}",
)
add_final_check(
    "Stage 2 row count and original order are exact",
    (
        len(stage2_scored) == int(READINESS_02["stage2_matchup_rows"])
        and stage2_scored["ID"].astype(str).reset_index(drop=True).equals(
            stage2_ids
        )
    ),
    f"rows={len(stage2_scored):,}",
)
add_final_check(
    "Stage 2 IDs are unique and lower-TeamID oriented",
    (
        stage2_scored["ID"].is_unique
        and stage2_scored["Team1ID"].lt(stage2_scored["Team2ID"]).all()
    ),
    "ID unique; Team1ID < Team2ID",
)
add_final_check(
    "all Stage 2 probabilities are finite and bounded",
    all(
        np.isfinite(stage2_scored[column]).all()
        and stage2_scored[column].between(0, 1).all()
        for column in ("PredPrimary", "PredChallenger", "PredBlend")
    ),
    "primary, challenger, and blend",
)
add_final_check(
    "every Stage 2 row has explicit primary and challenger routes",
    (
        stage2_scored["PrimaryRoute"].ne("").all()
        and stage2_scored["ChallengerRoute"].ne("").all()
    ),
    route_summary.to_json(orient="records"),
)
add_final_check(
    "seed-aware and seed-free routing is coherent",
    (
        stage2_scored.loc[
            stage2_scored["FeatureRoute"].eq("seed_aware"),
            "PrimaryRoute",
        ].str.endswith("_primary").all()
        and stage2_scored.loc[
            stage2_scored["FeatureRoute"].eq("seed_free"),
            "PrimaryRoute",
        ].str.endswith("_fallback").all()
    ),
    stage2_scored["FeatureRoute"].value_counts().to_dict().__repr__(),
)
add_final_check(
    "all submission files have exact schema and row count",
    all(
        list(frame.columns) == ["ID", "Pred"]
        and len(frame) == len(stage2_ids)
        and frame["ID"].astype(str).reset_index(drop=True).equals(stage2_ids)
        and frame["Pred"].between(0, 1).all()
        for frame in (
            submission_primary,
            submission_challenger,
            submission_blend,
        )
    ),
    submission_manifest.to_json(orient="records"),
)
missing_plot_files = [
    name
    for name in sorted(set(PLOT_ARTIFACTS))
    if not (FIGURE_DIR / name).exists()
]
add_final_check(
    "interactive Plotly reports were written",
    not missing_plot_files and len(set(PLOT_ARTIFACTS)) >= 10,
    f"reports={len(set(PLOT_ARTIFACTS))}; missing={missing_plot_files}",
)
add_final_check(
    "bracket audit passed when bracket artifacts were available",
    (not BRACKET_AVAILABLE) or (
        not bracket_audit.empty and bracket_audit["Passed"].all()
    ),
    "available" if BRACKET_AVAILABLE else "not available; nonblocking",
    blocking=False,
)

final_checks = pd.DataFrame(FINAL_CHECKS)
blocking_failures = final_checks.loc[
    final_checks["Blocking"] & ~final_checks["Passed"]
].copy()
atomic_csv(REPORT_DIR / "final_checks.csv", final_checks)

if not blocking_failures.empty:
    display(final_checks)
    display(blocking_failures)
    atomic_json(
        REPORT_DIR / "04_readiness_summary.json",
        {
            "status": "blocking_failures",
            "blocking_failures": blocking_failures["Check"].tolist(),
            "blueprint_sha256": FROZEN_BLUEPRINT["blueprint_sha256"],
            "recipe_sha256": MODEL_RECIPE["recipe_sha256"],
        },
    )
    raise AssertionError(
        "Notebook 04 has blocking integrity failures. Review "
        "reports/final_2026/final_checks.csv. Completed benchmark, model, "
        "and scoring checkpoints are preserved."
    )

# Employer-facing model card.
prequential_metrics = benchmark_metrics.loc[
    benchmark_metrics["Protocol"].eq("prequential_refit")
].copy()
metric_columns = [
    "Gender",
    "Role",
    "Architecture",
    "ModelDisplay",
    "MacroSeasonBrier",
    "GameWeightedBrier",
    "ROCAUC",
    "AveragePrecision",
    "AUCPR",
    "Precision",
    "Recall",
    "F1",
    "CalibrationSlope",
    "ECE",
]
model_card = f"""# NCAA Tournament Forecasting Model Card

## Purpose

Forecast every possible 2026 men's and women's tournament matchup as the probability that the lower-TeamID team wins.

## Information boundary

- Team features are frozen at or before Selection Sunday (DayNum 132).
- Tournament seasons are held out as indivisible evaluation groups.
- Model development ended in 2021.
- The locked 2022–2025 benchmark was consumed once under blueprint `{FROZEN_BLUEPRINT['blueprint_sha256']}`.
- Benchmark outcomes did not reopen feature engineering, hyperparameter selection, calibration-method selection, or model-family selection.

## Development comparison

Notebook 03 compared logistic regression, histogram boosting, XGBoost, LightGBM, PyTorch, TensorFlow, point-margin models, pooled models, separate models, constrained ensembles, and partial pooling under matched nested expanding-window validation.

### Development selections

- Men's primary: {STREAM_SPECS['M_primary']['architecture']} / {STREAM_SPECS['M_primary']['model_display']}
- Men's challenger: {STREAM_SPECS['M_challenger']['architecture']} / {STREAM_SPECS['M_challenger']['model_display']}
- Women's primary: {STREAM_SPECS['W_primary']['architecture']} / {STREAM_SPECS['W_primary']['model_display']}
- Women's challenger: {STREAM_SPECS['W_challenger']['architecture']} / {STREAM_SPECS['W_challenger']['model_display']}
- Development blend weights: {BLEND_WEIGHTS}

### Neural-model development audit

{neural_development.to_markdown(index=False)}

The neural systems were evaluated as bounded tabular MLP challengers. They were not promoted because their matched held-out-season Brier scores did not beat the frozen primary or performance-challenger streams.

## Locked benchmark

{prequential_metrics[metric_columns].to_markdown(index=False)}

## Final production routes

- Seed-aware rows use the development-frozen primary and challenger streams for the appropriate tournament.
- Seed-free hypothetical rows use a gender-specific elastic-net fallback.
- The default submission is the primary stream.
- Challenger and development-blend submissions are retained as transparent sensitivity products.

## Probability controls

- Algebraic matchup reversal is averaged at prediction time.
- Calibration methods are selected from development-only OOF predictions.
- Probability shrinkage and clipping are selected before the benchmark.
- Final calibration parameters are refit using out-of-sample predictions through 2025 without changing the calibration family.

## Limitations

- Tournament outcomes are sparse and season-to-season variance is material.
- Player injuries, roster continuity, betting-market information, and proprietary team ratings are not guaranteed to be present.
- Women's detailed box-score history begins later than men's and has documented early coverage gaps.
- Threshold metrics such as precision, recall, and F1 are secondary; the target is a calibrated probability and Brier score is primary.

## Reproducibility

- Recipe SHA-256: `{MODEL_RECIPE['recipe_sha256']}`
- Blueprint SHA-256: `{FROZEN_BLUEPRINT['blueprint_sha256']}`
- Historical feature store SHA-256: `{EXPECTED_CONTRACTS['historical_store']}`
- Stage 2 feature store SHA-256: `{EXPECTED_CONTRACTS['stage2_store']}`
- Primary submission SHA-256: `{sha256_file(SUBMISSION_PRIMARY_PATH)}`
"""
atomic_text(REPORT_DIR / "MODEL_CARD.md", model_card)

artifact_paths = {
    "model_recipe": RECIPE_PATH,
    "development_handoff_decisions": REPORT_DIR / "development_handoff_decisions.csv",
    "development_neural_model_audit": REPORT_DIR / "development_neural_model_audit.csv",
    "benchmark_blueprint": BLUEPRINT_PATH,
    "benchmark_lock": BENCHMARK_LOCK_PATH,
    "benchmark_predictions": BENCHMARK_PREDICTIONS_PATH,
    "benchmark_metrics": REPORT_DIR / "locked_benchmark_metrics.csv",
    "benchmark_slices": REPORT_DIR / "locked_benchmark_slices.csv",
    "benchmark_bootstrap": REPORT_DIR / "locked_benchmark_season_cluster_bootstrap.csv",
    "final_model_manifest": REPORT_DIR / "final_model_manifest.csv",
    "final_feature_importance": REPORT_DIR / "final_feature_importance.csv",
    "stage2_predictions": STAGE2_PREDICTIONS_PATH,
    "submission_primary": SUBMISSION_PRIMARY_PATH,
    "submission_challenger": SUBMISSION_CHALLENGER_PATH,
    "submission_blend": SUBMISSION_BLEND_PATH,
    "submission_default": SUBMISSION_DEFAULT_PATH,
    "model_card": REPORT_DIR / "MODEL_CARD.md",
    "final_checks": REPORT_DIR / "final_checks.csv",
}
if BRACKET_AVAILABLE:
    artifact_paths.update(
        {
            "championship_probabilities": REPORT_DIR / "championship_probabilities.csv",
            "bracket_audit": REPORT_DIR / "bracket_probability_audit.csv",
        }
    )
artifact_manifest = pd.DataFrame(
    [
        {
            "Artifact": name,
            "Path": str(path),
            "Exists": Path(path).exists(),
            "SizeMB": (
                round(Path(path).stat().st_size / 1024**2, 3)
                if Path(path).exists()
                else np.nan
            ),
            "SHA256": (
                sha256_file(Path(path)) if Path(path).exists() else None
            ),
            "RecipeSHA256": MODEL_RECIPE["recipe_sha256"],
            "BlueprintSHA256": FROZEN_BLUEPRINT["blueprint_sha256"],
        }
        for name, path in artifact_paths.items()
    ]
)
atomic_csv(REPORT_DIR / "artifact_manifest.csv", artifact_manifest)

readiness = {
    "notebook": "04_locked_benchmark_and_final_submission.ipynb",
    "notebook_implementation_version": NOTEBOOK_IMPLEMENTATION_VERSION,
    "status": "complete",
    "locked_benchmark_consumed": True,
    "locked_benchmark_seasons": list(LOCKED_SEASONS),
    "post_benchmark_retuning_permitted": False,
    "benchmark_protocols": sorted(benchmark_predictions["Protocol"].unique()),
    "benchmark_prediction_rows": int(len(benchmark_predictions)),
    "final_model_streams": sorted(FINAL_BUNDLES),
    "stage2_rows": int(len(stage2_scored)),
    "men_stage2_rows": int(stage2_scored["Gender"].eq("M").sum()),
    "women_stage2_rows": int(stage2_scored["Gender"].eq("W").sum()),
    "seed_aware_rows": int(stage2_scored["FeatureRoute"].eq("seed_aware").sum()),
    "seed_free_rows": int(stage2_scored["FeatureRoute"].eq("seed_free").sum()),
    "plotly_reports": sorted(set(PLOT_ARTIFACTS)),
    "blocking_final_check_failures": 0,
    "recipe_sha256": MODEL_RECIPE["recipe_sha256"],
    "blueprint_sha256": FROZEN_BLUEPRINT["blueprint_sha256"],
    "primary_submission": str(SUBMISSION_PRIMARY_PATH),
    "primary_submission_sha256": sha256_file(SUBMISSION_PRIMARY_PATH),
    "challenger_submission": str(SUBMISSION_CHALLENGER_PATH),
    "challenger_submission_sha256": sha256_file(SUBMISSION_CHALLENGER_PATH),
    "blend_submission": str(SUBMISSION_BLEND_PATH),
    "blend_submission_sha256": sha256_file(SUBMISSION_BLEND_PATH),
    "bracket_reporting_completed": bool(BRACKET_AVAILABLE),
}
atomic_json(REPORT_DIR / "04_readiness_summary.json", readiness)

log_event(
    "notebook_completed",
    status="complete",
    readiness=readiness,
)

display(final_checks)
print(
    "\nNOTEBOOK 04 COMPLETE — the locked benchmark was consumed without "
    "reopening model selection, final models were refit through 2025, "
    "Stage 2 was scored in restartable batches, and all submission audits passed."
)
print("Primary submission:", SUBMISSION_PRIMARY_PATH)
print("Challenger submission:", SUBMISSION_CHALLENGER_PATH)
print("Development blend submission:", SUBMISSION_BLEND_PATH)
print("Readiness report:", REPORT_DIR / "04_readiness_summary.json")


# Stop after the completion gate

The core forecasting pipeline is complete only after the final cell prints `NOTEBOOK 04 COMPLETE` and `reports/final_2026/04_readiness_summary.json` reports zero blocking failures.
